In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
!pip install kagglehub  # For Kaggle dataset access[object Object]


In [ ]:
import kagglehub
dataset_path = kagglehub.dataset_download("soumikrakshit/div2k-high-resolution-images")
print("DIV2K path:", dataset_path)


In [ ]:
import cv2
import os

# path returned from kagglehub
dataset_path = path

# folder containing HR images
hr_folder = os.path.join(dataset_path, "DIV2K_train_HR")

# pick one image
hr_path = os.path.join(hr_folder, os.listdir(hr_folder)[0])

# read image
hr = cv2.imread(hr_path)

scale = 4

# create LR image
lr = cv2.resize(hr, (hr.shape[1]//scale, hr.shape[0]//scale), interpolation=cv2.INTER_CUBIC)

print("HR shape:", hr.shape)
print("LR shape:", lr.shape)

In [ ]:
import cv2
hr = cv2.imread(hr_path)  # HWC BGR
lr = cv2.resize(hr, (hr.shape[1]//scale, hr.shape[0]//scale), interpolation=cv2.INTER_CUBIC)


In [ ]:
import numpy as np
noise_sigma = args.noise/255.0
lr = hr.astype(np.float32)/255.0 + np.random.normal(0, noise_sigma, hr.shape)


In [ ]:
encode_param = [int(cv2.IMWRITE_JPEG_QUALITY), args.jpeg]
_, encimg = cv2.imencode('.jpg', hr, encode_param)
lr = cv2.imdecode(encimg, cv2.IMREAD_COLOR)
from models.network_swinir import SwinIR
# Example: define a 4x SR model (RGB) with SwinIR-M (mid-size) parameters:
model = SwinIR(
    upscale=4, in_chans=3, img_size=64, window_size=8,
    img_range=1.0, depths=[6,6,6,6,6,6], embed_dim=180,
    num_heads=[6,6,6,6,6,6], mlp_ratio=2,
    upsampler='pixelshuffle', resi_connection='1conv'
)
from torch.utils.data import Dataset, DataLoader
from PIL import Image

class DIV2KDataset(Dataset):
    def __init__(self, root_dir, scale=4, task='classical_sr'):
        self.hr_paths = sorted(glob.glob(f"{root_dir}/train_HR/*.png"))
        self.scale = scale
        self.task = task

    def __len__(self):
        return len(self.hr_paths)

    def __getitem__(self, idx):
        hr = Image.open(self.hr_paths[idx]).convert('RGB')
        hr = np.array(hr).astype(np.float32) / 255.0
        # Generate LR on-the-fly
        if self.task.startswith('classical_sr'):
            lr = cv2.resize(hr, (hr.shape[1]//self.scale, hr.shape[0]//self.scale), interpolation=cv2.INTER_CUBIC)
        elif self.task == 'gray_dn':
            hr_gray = cv2.cvtColor(hr, cv2.COLOR_RGB2GRAY)
            noise = np.random.randn(*hr_gray.shape) * (noise_level/255.0)
            lr = np.clip(hr_gray + noise, 0, 1)[...,None]
            hr = hr_gray[...,None]
        # (Add JPEG case similarly)
        # Transpose to CHW
        lr = torch.from_numpy(lr.transpose(2,0,1)).float()
        hr = torch.from_numpy(hr.transpose(2,0,1)).float()
        return lr, hr

train_dataset = DIV2KDataset('DIV2K', scale=4, task='classical_sr')
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=4)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
loss_fn = torch.nn.L1Loss()
scaler = torch.cuda.amp.GradScaler()
for epoch in range(num_epochs):
    for lr_batch, hr_batch in train_loader:
        lr_batch, hr_batch = lr_batch.to(device), hr_batch.to(device)
        optimizer.zero_grad()
        with torch.cuda.amp.autocast():
            pred = model(lr_batch)
            loss = loss_fn(pred, hr_batch)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
if (epoch+1) % 10 == 0:
    torch.save(model.state_dict(), f"checkpoint_epoch{epoch+1}.pth")
_, _, H, W = img_lq.shape
h_pad = (H//window_size + 1) * window_size - H
w_pad = (W//window_size + 1) * window_size - W
img_lq = torch.cat([img_lq, torch.flip(img_lq, [2])], 2)[:,:,:H+h_pad,:]
img_lq = torch.cat([img_lq, torch.flip(img_lq, [3])], 3)[:,:,:,:W+w_pad]
output = model(img_lq)
output = output[..., :H*scale, :W*scale]  # crop
out_img = output.clamp(0,1).cpu().numpy().transpose(1,2,0)*255
cv2.imwrite("output.png", out_img[:,:,::-1])  # RGB to BGR
from utils.util_calculate_psnr_ssim import calculate_psnr, calculate_ssim, calculate_psnrb

psnr = calculate_psnr(output_img, gt_img, crop_border=scale)
ssim = calculate_ssim(output_img, gt_img, crop_border=scale)
if task in ['jpeg_car','color_jpeg_car']:
    psnrb = calculate_psnrb(output_img, gt_img, crop_border=scale)
!python swinir_div2k.py --task classical_sr --scale 4 --epochs 100 --batch_size 16 --lr 1e-4


In [ ]:
# ==============================
# 1. Install Dependencies
# ==============================
!pip install timm einops kagglehub -q

# ==============================
# 2. Imports
# ==============================
import os
import cv2
import torch
import numpy as np
import matplotlib.pyplot as plt
import kagglehub
from math import log10

# ==============================
# 3. Download Dataset
# ==============================
path = kagglehub.dataset_download("soumikrakshit/div2k-high-resolution-images")
print("Dataset Path:", path)

# ==============================
# 4. Find HR Folder (ROBUST)
# ==============================
def find_hr_folder(base_path):
    for root, dirs, files in os.walk(base_path):
        if "DIV2K_train_HR" in root:
            return root
    return None

hr_folder = find_hr_folder(path)

if hr_folder is None:
    raise Exception("❌ HR folder not found!")

print("✅ HR Folder:", hr_folder)

# ==============================
# 5. Find Images Recursively (FIXED)
# ==============================
def get_all_images(folder):
    image_paths = []
    for root, dirs, files in os.walk(folder):
        for file in files:
            if file.lower().endswith((".png", ".jpg", ".jpeg")):
                image_paths.append(os.path.join(root, file))
    return image_paths

images = get_all_images(hr_folder)

print("Total images found:", len(images))

if len(images) == 0:
    raise Exception("❌ No images found! Check dataset extraction.")

# pick first image
hr_path = images[0]
print("Using image:", hr_path)

# ==============================
# 6. Load Image Safely
# ==============================
hr = cv2.imread(hr_path)

if hr is None:
    raise Exception(f"❌ Failed to load image: {hr_path}")

hr = cv2.cvtColor(hr, cv2.COLOR_BGR2RGB)
print("✅ HR shape:", hr.shape)

# ==============================
# 7. Generate Low Resolution Image
# ==============================
scale = 4

lr = cv2.resize(
    hr,
    (hr.shape[1] // scale, hr.shape[0] // scale),
    interpolation=cv2.INTER_CUBIC
)

print("✅ LR shape:", lr.shape)

# ==============================
# 8. Simple SwinIR-like Model
# ==============================
import torch.nn as nn
import torch.nn.functional as F

class SimpleSwinIR(nn.Module):
    def __init__(self, upscale=4):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 64, 3, 1, 1)
        self.conv2 = nn.Conv2d(64, 64, 3, 1, 1)
        self.conv3 = nn.Conv2d(64, 3 * (upscale ** 2), 3, 1, 1)
        self.pixel_shuffle = nn.PixelShuffle(upscale)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = self.pixel_shuffle(self.conv3(x))
        return x

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SimpleSwinIR(upscale=4).to(device)
model.eval()

print("✅ Model running on:", device)

# ==============================
# 9. Prepare Tensor
# ==============================
lr_tensor = torch.from_numpy(lr / 255.).float().permute(2, 0, 1).unsqueeze(0).to(device)

# ==============================
# 10. Inference
# ==============================
with torch.no_grad():
    sr_tensor = model(lr_tensor)

sr = sr_tensor.squeeze().cpu().numpy().transpose(1, 2, 0)
sr = np.clip(sr, 0, 1)

# Resize to match HR
sr = cv2.resize(sr, (hr.shape[1], hr.shape[0]))

print("✅ SR shape:", sr.shape)

# ==============================
# 11. PSNR Calculation
# ==============================
def calculate_psnr(img1, img2):
    mse = np.mean((img1 - img2) ** 2)
    if mse == 0:
        return 100
    return 20 * log10(1.0 / np.sqrt(mse))

psnr = calculate_psnr(sr, hr / 255.)
print("📊 PSNR:", psnr)

# ==============================
# 12. Visualization
# ==============================
plt.figure(figsize=(15,5))

plt.subplot(1,3,1)
plt.title("High Resolution")
plt.imshow(hr)
plt.axis("off")

plt.subplot(1,3,2)
plt.title("Low Resolution")
plt.imshow(lr)
plt.axis("off")

plt.subplot(1,3,3)
plt.title("Restored Image")
plt.imshow(sr)
plt.axis("off")

plt.show()

In [ ]:
# ==============================
# 1. Install Dependencies
# ==============================
!pip install kagglehub -q

# ==============================
# 2. Imports
# ==============================
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import kagglehub
from math import log10

# ==============================
# 3. Download Dataset
# ==============================
path = kagglehub.dataset_download("soumikrakshit/div2k-high-resolution-images")
print("Dataset Path:", path)

# ==============================
# 4. Find HR Folder (ROBUST)
# ==============================
def find_hr_folder(base_path):
    for root, dirs, files in os.walk(base_path):
        if "DIV2K_train_HR" in root:
            return root
    return None

hr_folder = find_hr_folder(path)

if hr_folder is None:
    raise Exception("❌ HR folder not found!")

print("✅ HR Folder:", hr_folder)

# ==============================
# 5. Get Images Recursively
# ==============================
def get_all_images(folder):
    image_paths = []
    for root, dirs, files in os.walk(folder):
        for file in files:
            if file.lower().endswith((".png", ".jpg", ".jpeg")):
                image_paths.append(os.path.join(root, file))
    return image_paths

images = get_all_images(hr_folder)

if len(images) == 0:
    raise Exception("❌ No images found!")

print("Total images:", len(images))

# ==============================
# 6. Load Image
# ==============================
hr_path = images[0]
print("Using:", hr_path)

hr = cv2.imread(hr_path)

if hr is None:
    raise Exception("❌ Image loading failed!")

hr = cv2.cvtColor(hr, cv2.COLOR_BGR2RGB)
hr = hr / 255.0

print("✅ HR shape:", hr.shape)

# ==============================
# 7. Generate Low Resolution
# ==============================
scale = 4

lr = cv2.resize(
    hr,
    (hr.shape[1] // scale, hr.shape[0] // scale),
    interpolation=cv2.INTER_CUBIC
)

print("✅ LR shape:", lr.shape)

# ==============================
# 8. RESTORATION (FIXED)
# ==============================
# Using bicubic upsampling instead of untrained model

sr = cv2.resize(
    lr,
    (hr.shape[1], hr.shape[0]),
    interpolation=cv2.INTER_CUBIC
)

print("✅ SR shape:", sr.shape)

# ==============================
# 9. PSNR Calculation
# ==============================
def calculate_psnr(img1, img2):
    mse = np.mean((img1 - img2) ** 2)
    if mse == 0:
        return 100
    return 20 * log10(1.0 / np.sqrt(mse))

psnr = calculate_psnr(sr, hr)
print("📊 PSNR:", psnr)

# ==============================
# 10. Visualization
# ==============================
plt.figure(figsize=(15,5))

plt.subplot(1,3,1)
plt.title("High Resolution (GT)")
plt.imshow(hr)
plt.axis("off")

plt.subplot(1,3,2)
plt.title("Low Resolution")
plt.imshow(lr)
plt.axis("off")

plt.subplot(1,3,3)
plt.title("Restored (Bicubic)")
plt.imshow(sr)
plt.axis("off")

plt.show()

In [ ]:
# ==============================
# INSTALL DEPENDENCIES
# ==============================
!pip install timm einops kagglehub

# ==============================
# CLONE OFFICIAL SWINIR REPO
# ==============================
!git clone https://github.com/JingyunLiang/SwinIR.git

import sys
sys.path.append("./SwinIR")

# ==============================
# IMPORTS
# ==============================
import os
import cv2
import glob
import torch
import requests
import numpy as np
import matplotlib.pyplot as plt

from models.network_swinir import SwinIR

# ==============================
# DOWNLOAD DIV2K DATASET
# ==============================
import kagglehub

dataset_path = kagglehub.dataset_download("soumikrakshit/div2k-high-resolution-images")
hr_folder = os.path.join(dataset_path, "DIV2K_train_HR", "DIV2K_train_HR")

print("Dataset Path:", hr_folder)

# ==============================
# DOWNLOAD PRETRAINED MODEL
# ==============================
model_path = "003_realSR_BSRGAN_DFO_s64w8_SwinIR-M_x4_GAN.pth"

if not os.path.exists(model_path):
    print("Downloading pretrained SwinIR...")
    url = "https://github.com/JingyunLiang/SwinIR/releases/download/v0.0/" + model_path
    r = requests.get(url)
    open(model_path, "wb").write(r.content)

# ==============================
# DEFINE MODEL (MATCHES WEIGHTS)
# ==============================
model = SwinIR(
    upscale=4,
    in_chans=3,
    img_size=64,
    window_size=8,
    img_range=1.,
    depths=[6,6,6,6,6,6],
    embed_dim=180,
    num_heads=[6,6,6,6,6,6],
    mlp_ratio=2,
    upsampler='nearest+conv',
    resi_connection='1conv'
)

# ==============================
# LOAD WEIGHTS
# ==============================
pretrained = torch.load(model_path, map_location="cpu")

if "params_ema" in pretrained:
    pretrained = pretrained["params_ema"]

model.load_state_dict(pretrained, strict=True)

# ==============================
# DEVICE
# ==============================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()

print("Model loaded on:", device)

# ==============================
# GET IMAGES
# ==============================
images = sorted(glob.glob(os.path.join(hr_folder, "*.png")))

if len(images) == 0:
    raise Exception("❌ No images found!")

print("Total images:", len(images))

# ==============================
# OUTPUT FOLDER
# ==============================
os.makedirs("results", exist_ok=True)

# ==============================
# INFERENCE LOOP
# ==============================
for idx, hr_path in enumerate(images[:5]):  # limit for speed
    print(f"\nProcessing {idx+1}: {hr_path}")

    # read HR image
    hr = cv2.imread(hr_path)
    if hr is None:
        print("❌ Skipping bad image")
        continue

    hr = cv2.cvtColor(hr, cv2.COLOR_BGR2RGB)

    # create LR
    lr = cv2.resize(
        hr,
        (hr.shape[1] // 4, hr.shape[0] // 4),
        interpolation=cv2.INTER_CUBIC
    )

    # preprocess
    lr_tensor = torch.from_numpy(lr / 255.0).float().permute(2,0,1).unsqueeze(0).to(device)

    # inference
    with torch.no_grad():
        sr_tensor = model(lr_tensor)

    # postprocess
    sr = sr_tensor.squeeze().cpu().clamp(0,1).numpy()
    sr = np.transpose(sr, (1,2,0))
    sr = (sr * 255).astype(np.uint8)

    name = os.path.basename(hr_path).split('.')[0]

    # save images
    cv2.imwrite(f"results/{name}_LR.png", cv2.cvtColor(lr, cv2.COLOR_RGB2BGR))
    cv2.imwrite(f"results/{name}_SR.png", cv2.cvtColor(sr, cv2.COLOR_RGB2BGR))

print("\n✅ Processing Complete! Check /results")

# ==============================
# SHOW SAMPLE RESULT
# ==============================
sample = images[0]

hr = cv2.cvtColor(cv2.imread(sample), cv2.COLOR_BGR2RGB)
lr = cv2.resize(hr, (hr.shape[1]//4, hr.shape[0]//4), interpolation=cv2.INTER_CUBIC)
sr = cv2.cvtColor(cv2.imread(f"results/{os.path.basename(sample).split('.')[0]}_SR.png"), cv2.COLOR_BGR2RGB)

plt.figure(figsize=(12,4))

plt.subplot(1,3,1)
plt.title("HR")
plt.imshow(hr)
plt.axis("off")

plt.subplot(1,3,2)
plt.title("LR")
plt.imshow(lr)
plt.axis("off")

plt.subplot(1,3,3)
plt.title("SwinIR Output")
plt.imshow(sr)
plt.axis("off")

plt.show()

In [ ]:
# ==============================
# INSTALL DEPENDENCIES
# ==============================
!pip install timm einops kagglehub

# ==============================
# CLONE SWINIR REPO
# ==============================
!git clone https://github.com/JingyunLiang/SwinIR.git

import sys
sys.path.append("./SwinIR")

# ==============================
# IMPORTS
# ==============================
import os
import cv2
import glob
import torch
import requests
import numpy as np
import matplotlib.pyplot as plt

from models.network_swinir import SwinIR

# ==============================
# DOWNLOAD DIV2K DATASET
# ==============================
import kagglehub

dataset_path = kagglehub.dataset_download("soumikrakshit/div2k-high-resolution-images")
hr_folder = os.path.join(dataset_path, "DIV2K_train_HR", "DIV2K_train_HR")

print("Dataset Path:", hr_folder)

# ==============================
# DOWNLOAD PRETRAINED MODEL
# ==============================
model_path = "003_realSR_BSRGAN_DFO_s64w8_SwinIR-M_x4_GAN.pth"

if not os.path.exists(model_path):
    print("Downloading pretrained SwinIR...")
    url = "https://github.com/JingyunLiang/SwinIR/releases/download/v0.0/" + model_path
    r = requests.get(url)
    open(model_path, "wb").write(r.content)

# ==============================
# DEFINE MODEL (CORRECT CONFIG)
# ==============================
model = SwinIR(
    upscale=4,
    in_chans=3,
    img_size=64,
    window_size=8,
    img_range=1.,
    depths=[6,6,6,6,6,6],
    embed_dim=180,
    num_heads=[6,6,6,6,6,6],
    mlp_ratio=2,
    upsampler='nearest+conv',
    resi_connection='1conv'
)

# ==============================
# LOAD WEIGHTS
# ==============================
pretrained = torch.load(model_path, map_location="cpu")

if "params_ema" in pretrained:
    pretrained = pretrained["params_ema"]

model.load_state_dict(pretrained, strict=True)

# ==============================
# DEVICE SETUP
# ==============================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()

print("Model loaded on:", device)

# ==============================
# LOAD IMAGES
# ==============================
images = sorted(glob.glob(os.path.join(hr_folder, "*.png")))

if len(images) == 0:
    raise Exception("❌ No images found!")

print("Total images found:", len(images))

# ==============================
# OUTPUT DIRECTORY
# ==============================
os.makedirs("results", exist_ok=True)

# ==============================
# PSNR FUNCTION
# ==============================
def psnr(img1, img2):
    img1 = img1.astype(np.float32)
    img2 = img2.astype(np.float32)
    mse = np.mean((img1 - img2) ** 2)
    if mse == 0:
        return 100
    return 20 * np.log10(255.0 / np.sqrt(mse))

# ==============================
# INFERENCE ON 10 IMAGES
# ==============================
num_images = 10
results_data = []

for idx, hr_path in enumerate(images[:num_images]):
    print(f"\nProcessing {idx+1}: {hr_path}")

    hr = cv2.imread(hr_path)
    if hr is None:
        print("❌ Skipping bad image")
        continue

    hr = cv2.cvtColor(hr, cv2.COLOR_BGR2RGB)

    # create LR image
    lr = cv2.resize(
        hr,
        (hr.shape[1] // 4, hr.shape[0] // 4),
        interpolation=cv2.INTER_CUBIC
    )

    # preprocess
    lr_tensor = torch.from_numpy(lr / 255.0).float().permute(2,0,1).unsqueeze(0).to(device)

    # inference
    with torch.no_grad():
        sr_tensor = model(lr_tensor)

    # postprocess
    sr = sr_tensor.squeeze().cpu().clamp(0,1).numpy()
    sr = np.transpose(sr, (1,2,0))
    sr = (sr * 255).astype(np.uint8)

    # save images
    name = os.path.basename(hr_path).split('.')[0]
    cv2.imwrite(f"results/{name}_LR.png", cv2.cvtColor(lr, cv2.COLOR_RGB2BGR))
    cv2.imwrite(f"results/{name}_SR.png", cv2.cvtColor(sr, cv2.COLOR_RGB2BGR))

    # compute PSNR
    score = psnr(hr, sr)
    print(f"PSNR: {score:.2f} dB")

    results_data.append((hr, lr, sr, score))

print("\n✅ All images processed!")

# ==============================
# VISUALIZATION (10 IMAGES)
# ==============================
rows = len(results_data)
plt.figure(figsize=(12, 4 * rows))

for i, (hr, lr, sr, score) in enumerate(results_data):

    # HR
    plt.subplot(rows, 3, i*3 + 1)
    plt.imshow(hr)
    plt.title(f"HR {i+1}")
    plt.axis("off")

    # LR
    plt.subplot(rows, 3, i*3 + 2)
    plt.imshow(lr)
    plt.title(f"LR {i+1}")
    plt.axis("off")

    # SR
    plt.subplot(rows, 3, i*3 + 3)
    plt.imshow(sr)
    plt.title(f"SR {i+1}\nPSNR: {score:.2f}")
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# ==============================
# INSTALL DEPENDENCIES
# ==============================
!pip install timm einops kagglehub scikit-image

# ==============================
# CLONE SWINIR REPO
# ==============================
!git clone https://github.com/JingyunLiang/SwinIR.git

import sys
sys.path.append("./SwinIR")

# ==============================
# IMPORTS
# ==============================
import os
import cv2
import glob
import torch
import requests
import numpy as np
import matplotlib.pyplot as plt

from skimage.metrics import structural_similarity as ssim
from models.network_swinir import SwinIR

# ==============================
# DOWNLOAD MANGA109 DATASET
# ==============================
import kagglehub

dataset_path = kagglehub.dataset_download("yorha06/manga109")
print("Dataset Path:", dataset_path)

# get all images
images = glob.glob(os.path.join(dataset_path, "**/*.jpg"), recursive=True)

if len(images) == 0:
    raise Exception("❌ No images found!")

print("Total images found:", len(images))

# ==============================
# DOWNLOAD PRETRAINED MODEL
# ==============================
model_path = "003_realSR_BSRGAN_DFO_s64w8_SwinIR-M_x4_GAN.pth"

if not os.path.exists(model_path):
    print("Downloading pretrained model...")
    url = "https://github.com/JingyunLiang/SwinIR/releases/download/v0.0/" + model_path
    r = requests.get(url)
    open(model_path, "wb").write(r.content)

# ==============================
# DEFINE MODEL
# ==============================
model = SwinIR(
    upscale=4,
    in_chans=3,
    img_size=64,
    window_size=8,
    img_range=1.,
    depths=[6,6,6,6,6,6],
    embed_dim=180,
    num_heads=[6,6,6,6,6,6],
    mlp_ratio=2,
    upsampler='nearest+conv',
    resi_connection='1conv'
)

# ==============================
# LOAD WEIGHTS
# ==============================
pretrained = torch.load(model_path, map_location="cpu")

if "params_ema" in pretrained:
    pretrained = pretrained["params_ema"]

model.load_state_dict(pretrained, strict=True)

# ==============================
# DEVICE
# ==============================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()

print("Model loaded on:", device)

# ==============================
# METRICS
# ==============================
def calculate_psnr(img1, img2):
    img1 = img1.astype(np.float32)
    img2 = img2.astype(np.float32)
    mse = np.mean((img1 - img2) ** 2)
    if mse == 0:
        return 100
    return 20 * np.log10(255.0 / np.sqrt(mse))

def calculate_ssim(img1, img2):
    return ssim(img1, img2, channel_axis=2, data_range=255)

# ==============================
# OUTPUT DIRECTORY
# ==============================
os.makedirs("results", exist_ok=True)

# ==============================
# INFERENCE
# ==============================
num_images = 10
results_data = []
psnr_list = []
ssim_list = []

for idx, img_path in enumerate(images[:num_images]):

    print(f"\nProcessing {idx+1}: {img_path}")

    hr = cv2.imread(img_path)

    if hr is None:
        print("Skipping bad image")
        continue

    hr = cv2.cvtColor(hr, cv2.COLOR_BGR2RGB)

    # create LR
    lr = cv2.resize(
        hr,
        (hr.shape[1] // 4, hr.shape[0] // 4),
        interpolation=cv2.INTER_CUBIC
    )

    # preprocess
    lr_tensor = torch.from_numpy(lr / 255.0).float().permute(2,0,1).unsqueeze(0).to(device)

    # inference
    with torch.no_grad():
        sr_tensor = model(lr_tensor)

    # postprocess
    sr = sr_tensor.squeeze().cpu().clamp(0,1).numpy()
    sr = np.transpose(sr, (1,2,0))
    sr = (sr * 255).astype(np.uint8)

    # ==============================
    # FIX SHAPE MISMATCH (CRITICAL)
    # ==============================
    h, w = sr.shape[:2]
    hr_cropped = hr[:h, :w]

    # optional border crop (better evaluation)
    border = 4
    hr_crop = hr_cropped[border:h-border, border:w-border]
    sr_crop = sr[border:h-border, border:w-border]

    # ==============================
    # METRICS
    # ==============================
    psnr_val = calculate_psnr(hr_crop, sr_crop)
    ssim_val = calculate_ssim(hr_crop, sr_crop)

    print(f"PSNR: {psnr_val:.2f}, SSIM: {ssim_val:.4f}")

    psnr_list.append(psnr_val)
    ssim_list.append(ssim_val)

    # save images
    name = os.path.basename(img_path).split('.')[0]
    cv2.imwrite(f"results/{name}_LR.png", cv2.cvtColor(lr, cv2.COLOR_RGB2BGR))
    cv2.imwrite(f"results/{name}_SR.png", cv2.cvtColor(sr, cv2.COLOR_RGB2BGR))

    results_data.append((hr_cropped, lr, sr, psnr_val, ssim_val))

print("\n✅ Processing Complete!")

print(f"\nAverage PSNR: {np.mean(psnr_list):.2f}")
print(f"Average SSIM: {np.mean(ssim_list):.4f}")

# ==============================
# VISUALIZATION
# ==============================
rows = len(results_data)
plt.figure(figsize=(12, 4 * rows))

for i, (hr, lr, sr, psnr_val, ssim_val) in enumerate(results_data):

    plt.subplot(rows, 3, i*3 + 1)
    plt.imshow(hr)
    plt.title(f"HR {i+1}")
    plt.axis("off")

    plt.subplot(rows, 3, i*3 + 2)
    plt.imshow(lr)
    plt.title(f"LR {i+1}")
    plt.axis("off")

    plt.subplot(rows, 3, i*3 + 3)
    plt.imshow(sr)
    plt.title(f"SR {i+1}\nPSNR:{psnr_val:.2f} SSIM:{ssim_val:.3f}")
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# ==============================
# INSTALL DEPENDENCIES
# ==============================
!pip install timm einops kagglehub scikit-image

# ==============================
# CLONE SWINIR
# ==============================
!git clone https://github.com/JingyunLiang/SwinIR.git

import sys, os, random, glob
sys.path.append("./SwinIR")

import cv2
import torch
import numpy as np
import matplotlib.pyplot as plt

from skimage.metrics import structural_similarity as ssim
from models.network_swinir import SwinIR

# ==============================
# DOWNLOAD DATASET
# ==============================
import kagglehub
dataset_path = kagglehub.dataset_download("yorha06/manga109")

images = glob.glob(os.path.join(dataset_path, "**/*.jpg"), recursive=True)
print("Total images:", len(images))

# ==============================
# SPLIT DATA
# ==============================
random.shuffle(images)
train_imgs = images[:200]
test_imgs  = images[200:210]

# ==============================
# PATCH FUNCTION
# ==============================
def get_patch(img, patch_size=64, scale=4):
    h, w = img.shape[:2]

    if h < patch_size or w < patch_size:
        return None, None

    x = random.randint(0, w - patch_size)
    y = random.randint(0, h - patch_size)

    hr_patch = img[y:y+patch_size, x:x+patch_size]
    lr_patch = cv2.resize(hr_patch, (patch_size//scale, patch_size//scale), interpolation=cv2.INTER_CUBIC)

    return hr_patch, lr_patch

# ==============================
# MODEL (LIGHTWEIGHT SWINIR)
# ==============================
model = SwinIR(
    upscale=4,
    in_chans=3,
    img_size=64,
    window_size=8,
    img_range=1.,
    depths=[4,4,4,4],
    embed_dim=60,
    num_heads=[6,6,6,6],
    mlp_ratio=2,
    upsampler='pixelshuffle',
    resi_connection='1conv'
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# ==============================
# TRAIN SETUP
# ==============================
criterion = torch.nn.L1Loss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

# ==============================
# TRAINING (STABLE)
# ==============================
print("\n🚀 Training...\n")

epochs = 2

for epoch in range(epochs):
    model.train()
    total_loss = 0
    valid_steps = 0

    for img_path in train_imgs:

        hr = cv2.imread(img_path)
        if hr is None:
            continue

        hr = cv2.cvtColor(hr, cv2.COLOR_BGR2RGB)

        if hr.shape[0] < 64 or hr.shape[1] < 64:
            continue

        hr_patch, lr_patch = get_patch(hr)
        if hr_patch is None:
            continue

        hr_t = torch.from_numpy(hr_patch/255.).float().permute(2,0,1).unsqueeze(0).to(device)
        lr_t = torch.from_numpy(lr_patch/255.).float().permute(2,0,1).unsqueeze(0).to(device)

        sr = model(lr_t)
        loss = criterion(sr, hr_t)

        if torch.isnan(loss):
            continue

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        valid_steps += 1

    print(f"Epoch {epoch+1}, Loss: {total_loss/max(1,valid_steps):.4f}")

print("\n✅ Training Done!")

# ==============================
# TILE INFERENCE (NO OOM)
# ==============================
def inference_tile(model, lr, tile=128, scale=4):
    h, w, c = lr.shape
    sr = np.zeros((h*scale, w*scale, c), dtype=np.float32)

    for y in range(0, h, tile):
        for x in range(0, w, tile):

            patch = lr[y:y+tile, x:x+tile]

            patch_t = torch.from_numpy(patch/255.).float().permute(2,0,1).unsqueeze(0).to(device)

            with torch.no_grad():
                out = model(patch_t)

            out = out.squeeze().cpu().numpy().transpose(1,2,0)

            sr[y*scale:(y+tile)*scale, x*scale:(x+tile)*scale] = out

    return (sr * 255).clip(0,255).astype(np.uint8)

# ==============================
# METRICS
# ==============================
def psnr(img1, img2):
    mse = np.mean((img1 - img2) ** 2)
    return 100 if mse == 0 else 20*np.log10(255.0/np.sqrt(mse))

def ssim_calc(img1, img2):
    return ssim(img1, img2, channel_axis=2, data_range=255)

# ==============================
# TESTING
# ==============================
model.eval()

results = []
psnr_vals = []
ssim_vals = []

print("\n🧪 Testing...\n")

for img_path in test_imgs:

    hr = cv2.imread(img_path)
    if hr is None:
        continue

    hr = cv2.cvtColor(hr, cv2.COLOR_BGR2RGB)

    h, w = hr.shape[:2]
    h = h - h % 4
    w = w - w % 4
    hr = hr[:h, :w]

    lr = cv2.resize(hr, (w//4, h//4), interpolation=cv2.INTER_CUBIC)

    sr = inference_tile(model, lr)

    h2, w2 = sr.shape[:2]
    hr = hr[:h2, :w2]

    # crop border (important)
    border = 4
    hr_crop = hr[border:h2-border, border:w2-border]
    sr_crop = sr[border:h2-border, border:w2-border]

    p = psnr(hr_crop, sr_crop)
    s = ssim_calc(hr_crop, sr_crop)

    psnr_vals.append(p)
    ssim_vals.append(s)

    results.append((hr, lr, sr, p, s))

    print(f"PSNR: {p:.2f}, SSIM: {s:.4f}")

print("\n📊 Average PSNR:", np.mean(psnr_vals))
print("📊 Average SSIM:", np.mean(ssim_vals))

# ==============================
# VISUALIZATION (10 IMAGES)
# ==============================
plt.figure(figsize=(12, 4*len(results)))

for i, (hr, lr, sr, p, s) in enumerate(results):

    plt.subplot(len(results),3,i*3+1)
    plt.imshow(hr)
    plt.title("HR")
    plt.axis("off")

    plt.subplot(len(results),3,i*3+2)
    plt.imshow(lr)
    plt.title("LR")
    plt.axis("off")

    plt.subplot(len(results),3,i*3+3)
    plt.imshow(sr)
    plt.title(f"SR\nPSNR:{p:.2f} SSIM:{s:.3f}")
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# ==============================
# INSTALL DEPENDENCIES
# ==============================
!pip install timm einops kagglehub scikit-image

# ==============================
# CLONE SWINIR
# ==============================
!git clone https://github.com/JingyunLiang/SwinIR.git

import sys, os, random, glob
sys.path.append("./SwinIR")

import cv2
import torch
import numpy as np
import matplotlib.pyplot as plt

from skimage.metrics import structural_similarity as ssim
from models.network_swinir import SwinIR

# ==============================
# DOWNLOAD DATASET
# ==============================
import kagglehub
dataset_path = kagglehub.dataset_download("soumikrakshit/div2k-high-resolution-images")

images = glob.glob(os.path.join(dataset_path, "**/*.png"), recursive=True)
if len(images) == 0:
    images = glob.glob(os.path.join(dataset_path, "**/*.jpg"), recursive=True)

print("Total images:", len(images))

# ==============================
# SPLIT DATA
# ==============================
random.shuffle(images)

train_imgs = images[:800]     # 🔥 increased data
test_imgs  = images[800:810]

# ==============================
# DATA AUGMENTATION
# ==============================
def augment(img):
    if random.random() > 0.5:
        img = cv2.flip(img, 1)
    if random.random() > 0.5:
        img = cv2.flip(img, 0)
    if random.random() > 0.5:
        img = np.rot90(img)
    return img

# ==============================
# PATCH FUNCTION
# ==============================
def get_patch(img, patch_size=64, scale=4):
    h, w = img.shape[:2]

    if h < patch_size or w < patch_size:
        return None, None

    x = random.randint(0, w - patch_size)
    y = random.randint(0, h - patch_size)

    hr = img[y:y+patch_size, x:x+patch_size]
    hr = augment(hr)

    # 🔥 better degradation (bicubic)
    lr = cv2.resize(hr, (patch_size//scale, patch_size//scale), interpolation=cv2.INTER_CUBIC)

    return hr, lr

# ==============================
# MODEL (IMPROVED)
# ==============================
model = SwinIR(
    upscale=4,
    in_chans=3,
    img_size=64,
    window_size=8,
    img_range=1.,
    depths=[6,6,6,6],      # 🔥 deeper
    embed_dim=96,          # 🔥 stronger
    num_heads=[6,6,6,6],
    mlp_ratio=2,
    upsampler='pixelshuffle',
    resi_connection='1conv'
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# ==============================
# TRAIN SETUP
# ==============================
criterion = torch.nn.L1Loss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

# ==============================
# TRAINING
# ==============================
print("\n🚀 Training...\n")

epochs = 30

for epoch in range(epochs):
    model.train()
    total_loss = 0
    steps = 0

    for img_path in train_imgs:

        hr = cv2.imread(img_path)
        if hr is None:
            continue

        hr = cv2.cvtColor(hr, cv2.COLOR_BGR2RGB)

        hr_patch, lr_patch = get_patch(hr)
        if hr_patch is None:
            continue

        hr_t = torch.from_numpy(hr_patch/255.).float().permute(2,0,1).unsqueeze(0).to(device)
        lr_t = torch.from_numpy(lr_patch/255.).float().permute(2,0,1).unsqueeze(0).to(device)

        sr = model(lr_t)
        loss = criterion(sr, hr_t)

        optimizer.zero_grad()
        loss.backward()

        # 🔥 stability improvement
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        optimizer.step()

        total_loss += loss.item()
        steps += 1

    print(f"Epoch {epoch+1}, Loss: {total_loss/max(1,steps):.4f}")

print("\n✅ Training Done!")

# ==============================
# INFERENCE
# ==============================
def inference(model, lr):
    lr_t = torch.from_numpy(lr/255.).float().permute(2,0,1).unsqueeze(0).to(device)

    with torch.no_grad():
        sr = model(lr_t)

    sr = sr.squeeze().cpu().numpy().transpose(1,2,0)
    return (sr * 255).clip(0,255).astype(np.uint8)

# ==============================
# METRICS
# ==============================
def psnr(img1, img2):
    h = min(img1.shape[0], img2.shape[0])
    w = min(img1.shape[1], img2.shape[1])

    img1 = img1[:h, :w].astype(np.float32)
    img2 = img2[:h, :w].astype(np.float32)

    mse = np.mean((img1 - img2) ** 2)
    return 100 if mse == 0 else 20*np.log10(255.0/np.sqrt(mse))

def ssim_calc(img1, img2):
    h = min(img1.shape[0], img2.shape[0])
    w = min(img1.shape[1], img2.shape[1])

    return ssim(img1[:h,:w], img2[:h,:w], channel_axis=2, data_range=255)

# ==============================
# TESTING
# ==============================
model.eval()

psnr_vals, ssim_vals = [], []
bicubic_psnr = []

results = []

print("\n🧪 Testing...\n")

for img_path in test_imgs:

    hr = cv2.imread(img_path)
    if hr is None:
        continue

    hr = cv2.cvtColor(hr, cv2.COLOR_BGR2RGB)

    h, w = hr.shape[:2]
    h -= h % 4
    w -= w % 4
    hr = hr[:h, :w]

    # LR
    lr = cv2.resize(hr, (w//4, h//4), interpolation=cv2.INTER_CUBIC)

    # Bicubic baseline
    bicubic = cv2.resize(lr, (w, h), interpolation=cv2.INTER_CUBIC)

    # Model SR
    sr = inference(model, lr)

    p = psnr(hr, sr)
    s = ssim_calc(hr, sr)

    p_bic = psnr(hr, bicubic)

    psnr_vals.append(p)
    ssim_vals.append(s)
    bicubic_psnr.append(p_bic)

    results.append((hr, lr, sr, bicubic, p, s))

    print(f"SwinIR PSNR: {p:.2f} | Bicubic: {p_bic:.2f} | SSIM: {s:.4f}")

print("\n📊 Avg SwinIR PSNR:", np.mean(psnr_vals))
print("📊 Avg Bicubic PSNR:", np.mean(bicubic_psnr))
print("📊 Avg SSIM:", np.mean(ssim_vals))

# ==============================
# VISUALIZATION
# ==============================
plt.figure(figsize=(15, 5*len(results)))

for i, (hr, lr, sr, bicubic, p, s) in enumerate(results):

    plt.subplot(len(results),4,i*4+1)
    plt.imshow(hr)
    plt.title("HR")
    plt.axis("off")

    plt.subplot(len(results),4,i*4+2)
    plt.imshow(lr)
    plt.title("LR")
    plt.axis("off")

    plt.subplot(len(results),4,i*4+3)
    plt.imshow(bicubic)
    plt.title("Bicubic")
    plt.axis("off")

    plt.subplot(len(results),4,i*4+4)
    plt.imshow(sr)
    plt.title(f"SwinIR\nPSNR:{p:.2f} SSIM:{s:.3f}")
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# ==============================
# INSTALL DEPENDENCIES
# ==============================
# !pip install timm einops kagglehub scikit-image

# ==============================
# CLONE SWINIR
# ==============================
# !git clone https://github.com/JingyunLiang/SwinIR.git

import sys, os, random, glob, math, time
sys.path.append("./SwinIR")

import cv2
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import CosineAnnealingLR
from skimage.metrics import structural_similarity as ssim
from models.network_swinir import SwinIR

# ==============================
# DOWNLOAD DATASETS
# ==============================
import kagglehub

print("Downloading DIV2K...")
div2k_path = kagglehub.dataset_download("soumikrakshit/div2k-high-resolution-images")

print("Downloading Flickr2K...")
flickr2k_path = kagglehub.dataset_download("daehoyang/flickr2k")

# ==============================
# COLLECT ALL IMAGES
# ==============================
def collect_images(root):
    imgs = glob.glob(os.path.join(root, "**/*.png"), recursive=True)
    if not imgs:
        imgs = glob.glob(os.path.join(root, "**/*.jpg"), recursive=True)
    return sorted(imgs)

div2k_imgs    = collect_images(div2k_path)
flickr2k_imgs = collect_images(flickr2k_path)

print(f"DIV2K images   : {len(div2k_imgs)}")
print(f"Flickr2K images: {len(flickr2k_imgs)}")

# ==============================
# SPLIT DATA
# ==============================
random.seed(42)
random.shuffle(div2k_imgs)
test_imgs  = div2k_imgs[:10]
train_imgs = div2k_imgs[10:] + flickr2k_imgs

print(f"Train: {len(train_imgs)} | Test: {len(test_imgs)}")

# ==============================
# HYPERPARAMETERS
# ==============================
PATCH_SIZE  = 64
SCALE       = 4
BATCH_SIZE  = 4      # lower batch = less RAM, safer on Kaggle/Colab
PATCHES_PER = 4      # patches per image per epoch (halved vs before to cut epoch time)
EPOCHS      = 3
LR          = 2e-4
WINDOW_SIZE = 8

# ── CRITICAL for Jupyter / Kaggle / Colab ─────────────────────────────────────
# multiprocessing DataLoader workers deadlock inside ipykernel.
# Always keep num_workers=0 in notebooks.
NUM_WORKERS = 0
# ──────────────────────────────────────────────────────────────────────────────

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# ==============================
# DATA AUGMENTATION
# ==============================
def augment(hr):
    if random.random() > 0.5:
        hr = cv2.flip(hr, 1)
    if random.random() > 0.5:
        hr = cv2.flip(hr, 0)
    k = random.randint(0, 3)
    if k > 0:
        hr = np.rot90(hr, k).copy()   # .copy() avoids negative-stride tensor error
    return hr

# ==============================
# DATASET
# ==============================
class SRDataset(Dataset):
    def __init__(self, img_paths, patch_size=PATCH_SIZE, scale=SCALE,
                 patches_per=PATCHES_PER):
        self.paths      = img_paths
        self.patch_size = patch_size
        self.scale      = scale
        self.ppp        = patches_per

    def __len__(self):
        return len(self.paths) * self.ppp

    def __getitem__(self, idx):
        lr_size = self.patch_size // self.scale
        zero_lr = torch.zeros(3, lr_size,         lr_size)
        zero_hr = torch.zeros(3, self.patch_size, self.patch_size)

        img_path = self.paths[idx % len(self.paths)]
        hr = cv2.imread(img_path)
        if hr is None:
            return zero_lr, zero_hr

        hr = cv2.cvtColor(hr, cv2.COLOR_BGR2RGB)
        h, w = hr.shape[:2]

        if h < self.patch_size or w < self.patch_size:
            return zero_lr, zero_hr

        x        = random.randint(0, w - self.patch_size)
        y        = random.randint(0, h - self.patch_size)
        hr_patch = hr[y:y+self.patch_size, x:x+self.patch_size]
        hr_patch = augment(hr_patch)

        lr_patch = cv2.resize(hr_patch, (lr_size, lr_size),
                              interpolation=cv2.INTER_CUBIC)

        hr_t = torch.from_numpy(hr_patch.astype(np.float32) / 255.).permute(2, 0, 1)
        lr_t = torch.from_numpy(lr_patch.astype(np.float32) / 255.).permute(2, 0, 1)
        return lr_t, hr_t


train_dataset = SRDataset(train_imgs)
train_loader  = DataLoader(
    train_dataset,
    batch_size  = BATCH_SIZE,
    shuffle     = True,
    num_workers = NUM_WORKERS,
    pin_memory  = (device.type == 'cuda'),
    drop_last   = True,
)
print(f"Steps per epoch: {len(train_loader)}")

# ==============================
# MODEL
# Choose 'standard' (paper-spec, 11.8 M params) for best quality,
# or 'lightweight' (faster, ~900 K params) for quick experiments.
# ==============================
MODEL_SIZE = 'standard'

if MODEL_SIZE == 'standard':
    model = SwinIR(
        upscale=SCALE, in_chans=3, img_size=PATCH_SIZE,
        window_size=WINDOW_SIZE, img_range=1.,
        depths=[6, 6, 6, 6, 6, 6], embed_dim=180,
        num_heads=[6, 6, 6, 6, 6, 6], mlp_ratio=2,
        upsampler='pixelshuffle', resi_connection='3conv'
    ).to(device)
else:
    model = SwinIR(
        upscale=SCALE, in_chans=3, img_size=PATCH_SIZE,
        window_size=WINDOW_SIZE, img_range=1.,
        depths=[4, 4, 4, 4], embed_dim=60,
        num_heads=[6, 6, 6, 6], mlp_ratio=2,
        upsampler='pixelshuffle', resi_connection='1conv'
    ).to(device)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model ({MODEL_SIZE}): {total_params/1e6:.1f}M parameters")

# ==============================
# LOSS / OPTIMISER / SCHEDULER
# ==============================
criterion = nn.L1Loss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR, betas=(0.9, 0.99))
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-7)

# ── Modern AMP API — no FutureWarning ─────────────────────────────────────────
use_amp = (device.type == 'cuda')
scaler  = torch.amp.GradScaler('cuda', enabled=use_amp)
# ──────────────────────────────────────────────────────────────────────────────

# ==============================
# TRAINING LOOP
# ==============================
print("\n🚀 Training started...\n")

best_loss    = float('inf')
loss_history = []
LOG_EVERY    = max(1, len(train_loader) // 5)   # print ~5 times per epoch

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0
    steps      = 0
    t0         = time.time()

    for batch_idx, (lr_batch, hr_batch) in enumerate(train_loader):
        lr_batch = lr_batch.to(device, non_blocking=True)
        hr_batch = hr_batch.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast('cuda', enabled=use_amp):
            sr_batch = model(lr_batch)
            loss     = criterion(sr_batch, hr_batch)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
        steps      += 1

        # per-step heartbeat so you know the kernel is alive
        if (batch_idx + 1) % LOG_EVERY == 0:
            print(f"  Ep {epoch:3d} | step {batch_idx+1:5d}/{len(train_loader)} "
                  f"| loss {loss.item():.4f} | {time.time()-t0:.0f}s")

    scheduler.step()
    avg_loss = total_loss / max(1, steps)
    loss_history.append(avg_loss)

    print(f"Epoch [{epoch:3d}/{EPOCHS}] avg loss {avg_loss:.4f} "
          f"lr {scheduler.get_last_lr()[0]:.2e} "
          f"time {time.time()-t0:.1f}s")

    if avg_loss < best_loss:
        best_loss = avg_loss
        torch.save(model.state_dict(), "swinir_best.pth")
        print(f"  ✅ Saved checkpoint (loss {best_loss:.4f})")

print(f"\n✅ Training complete. Best loss: {best_loss:.4f}")

# ==============================
# INFERENCE  (with window-size padding)
# ==============================
def inference(model, lr_img):
    h, w  = lr_img.shape[:2]
    pad_h = (WINDOW_SIZE - h % WINDOW_SIZE) % WINDOW_SIZE
    pad_w = (WINDOW_SIZE - w % WINDOW_SIZE) % WINDOW_SIZE

    lr_pad = np.pad(lr_img, ((0, pad_h), (0, pad_w), (0, 0)), mode='reflect')
    lr_t   = (torch.from_numpy(lr_pad.astype(np.float32) / 255.)
              .permute(2, 0, 1).unsqueeze(0).to(device))

    with torch.no_grad():
        with torch.amp.autocast('cuda', enabled=use_amp):
            sr_t = model(lr_t)

    sr = sr_t.squeeze().float().cpu().numpy().transpose(1, 2, 0)
    sr = (sr * 255).clip(0, 255).astype(np.uint8)
    return sr[:h * SCALE, :w * SCALE]

# ==============================
# METRICS
# ==============================
def calc_psnr(img1, img2):
    h = min(img1.shape[0], img2.shape[0])
    w = min(img1.shape[1], img2.shape[1])
    a = img1[:h, :w].astype(np.float32)
    b = img2[:h, :w].astype(np.float32)
    mse = np.mean((a - b) ** 2)
    return float('inf') if mse == 0 else 20 * np.log10(255. / np.sqrt(mse))

def calc_ssim(img1, img2):
    h = min(img1.shape[0], img2.shape[0])
    w = min(img1.shape[1], img2.shape[1])
    return ssim(img1[:h, :w], img2[:h, :w], channel_axis=2, data_range=255)

# ==============================
# EVALUATION
# ==============================
model.eval()
results = []
print("\n🧪 Evaluating on test set...\n")

for img_path in test_imgs:
    hr = cv2.imread(img_path)
    if hr is None:
        continue

    hr    = cv2.cvtColor(hr, cv2.COLOR_BGR2RGB)
    align = SCALE * WINDOW_SIZE
    h, w  = hr.shape[:2]
    hr    = hr[:(h//align)*align, :(w//align)*align]
    h, w  = hr.shape[:2]

    lr      = cv2.resize(hr, (w//SCALE, h//SCALE), interpolation=cv2.INTER_CUBIC)
    bicubic = cv2.resize(lr, (w, h),               interpolation=cv2.INTER_CUBIC)
    sr      = inference(model, lr)

    p_sr  = calc_psnr(hr, sr)
    s_sr  = calc_ssim(hr, sr)
    p_bic = calc_psnr(hr, bicubic)

    results.append({
        "hr": hr, "lr": lr, "sr": sr, "bicubic": bicubic,
        "psnr_sr": p_sr, "ssim_sr": s_sr,
        "psnr_bic": p_bic, "gain": p_sr - p_bic,
        "path": img_path,
    })
    print(f"  {os.path.basename(img_path):<30s} "
          f"SwinIR: {p_sr:.2f} dB | Bicubic: {p_bic:.2f} dB | "
          f"+{p_sr-p_bic:.2f} dB | SSIM: {s_sr:.4f}")

print(f"\n📊 Avg SwinIR PSNR : {np.mean([r['psnr_sr']  for r in results]):.2f} dB")
print(f"📊 Avg Bicubic PSNR: {np.mean([r['psnr_bic'] for r in results]):.2f} dB")
print(f"📊 Avg PSNR Gain   : +{np.mean([r['gain']     for r in results]):.2f} dB")
print(f"📊 Avg SSIM        : {np.mean([r['ssim_sr']  for r in results]):.4f}")

# ==============================
# TOP-10 SELECTION
# ==============================
top_k = min(10, len(results))
top10 = sorted(results, key=lambda x: x["psnr_sr"], reverse=True)[:top_k]

print(f"\n🏆 Top-{top_k} by SwinIR PSNR:")
for i, r in enumerate(top10, 1):
    print(f"  #{i:2d}  {os.path.basename(r['path']):<28s} "
          f"PSNR {r['psnr_sr']:.2f} | Bicubic {r['psnr_bic']:.2f} | "
          f"+{r['gain']:.2f} dB | SSIM {r['ssim_sr']:.4f}")

# ==============================
# HELPER
# ==============================
def centre_crop(img, size=320):
    h, w = img.shape[:2]
    s    = min(h, w, size)
    return img[(h-s)//2:(h-s)//2+s, (w-s)//2:(w-s)//2+s]

# ==============================
# FIGURE 1 — Loss curve + all test results
# ==============================
n    = len(results)
fig1, axes1 = plt.subplots(n + 1, 4, figsize=(16, 5*(n+1)))
fig1.suptitle("SwinIR — Training Loss & Test Results", fontsize=13)

axes1[0,0].plot(range(1, EPOCHS+1), loss_history, color='steelblue', lw=1.5)
axes1[0,0].set_title("Training Loss (L1)")
axes1[0,0].set_xlabel("Epoch"); axes1[0,0].set_ylabel("Loss")
axes1[0,0].grid(alpha=0.3)
for ax in axes1[0, 1:]: ax.axis('off')

for i, r in enumerate(results):
    for ax, img, title in zip(
        axes1[i+1],
        [r["hr"], r["lr"], r["bicubic"], r["sr"]],
        ["HR", "LR",
         f"Bicubic\n{r['psnr_bic']:.2f} dB",
         f"SwinIR\n{r['psnr_sr']:.2f} dB  SSIM:{r['ssim_sr']:.3f}"]
    ):
        ax.imshow(img); ax.set_title(title, fontsize=9); ax.axis('off')

plt.tight_layout()
plt.savefig("swinir_all_results.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved → swinir_all_results.png")

# ==============================
# FIGURE 2 — Top-10 visual grid
# ==============================
fig2, axes2 = plt.subplots(top_k, 4, figsize=(18, 4.5*top_k))
fig2.suptitle(f"Top-{top_k} Images by SwinIR PSNR  (×{SCALE} SR)",
              fontsize=14, fontweight='bold')

for ax, ct in zip(axes2[0],
                  ["HR (Ground Truth)", "LR (Input)",
                   "Bicubic Baseline",  "SwinIR Output"]):
    ax.set_title(ct, fontsize=11, fontweight='bold', pad=8)

for rank, r in enumerate(top10):
    for col, img in enumerate([r["hr"], r["lr"], r["bicubic"], r["sr"]]):
        ax = axes2[rank][col]
        ax.imshow(centre_crop(img))
        ax.axis('off')

        if col == 0:
            ax.text(0.02, 0.97,
                    f"#{rank+1}  {os.path.basename(r['path'])}",
                    transform=ax.transAxes, fontsize=7.5,
                    color='white', va='top',
                    bbox=dict(boxstyle='round,pad=0.3',
                              facecolor='black', alpha=0.55))
        if col == 2:
            ax.text(0.02, 0.97, f"PSNR: {r['psnr_bic']:.2f} dB",
                    transform=ax.transAxes, fontsize=7.5,
                    color='white', va='top',
                    bbox=dict(boxstyle='round,pad=0.3',
                              facecolor='#555', alpha=0.65))
        if col == 3:
            ax.text(0.02, 0.97,
                    f"PSNR: {r['psnr_sr']:.2f} dB\n"
                    f"vs Bicubic: {r['psnr_bic']:.2f} dB\n"
                    f"Gain: +{r['gain']:.2f} dB\n"
                    f"SSIM: {r['ssim_sr']:.4f}",
                    transform=ax.transAxes, fontsize=7.5,
                    color='white', va='top',
                    bbox=dict(boxstyle='round,pad=0.3',
                              facecolor='#1a6e3c', alpha=0.75))

plt.tight_layout()
plt.savefig("swinir_top10_psnr.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved → swinir_top10_psnr.png")

# ==============================
# FIGURE 3 — PSNR bar chart
# ==============================
fig3, ax3 = plt.subplots(figsize=(12, 5))
names = [f"#{i+1} {os.path.basename(r['path'])[:18]}" for i, r in enumerate(top10)]
p_sw  = [r["psnr_sr"]  for r in top10]
p_bic = [r["psnr_bic"] for r in top10]
x     = np.arange(top_k)
bw    = 0.35

b1 = ax3.bar(x - bw/2, p_bic, bw, label='Bicubic', color='#888780', alpha=0.85)
b2 = ax3.bar(x + bw/2, p_sw,  bw, label='SwinIR',  color='#3266ad', alpha=0.85)

for bar, ps, pb in zip(b2, p_sw, p_bic):
    ax3.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 0.05,
             f"+{ps-pb:.2f}", ha='center', va='bottom',
             fontsize=8, color='#1a5fa8', fontweight='bold')

ax3.set_xticks(x)
ax3.set_xticklabels(names, rotation=30, ha='right', fontsize=9)
ax3.set_ylabel("PSNR (dB)")
ax3.set_title(f"Top-{top_k} — SwinIR vs Bicubic PSNR", fontsize=12)
ax3.legend(); ax3.grid(axis='y', alpha=0.3)
ax3.set_ylim(min(p_bic) - 1, max(p_sw) + 1.5)

plt.tight_layout()
plt.savefig("swinir_top10_bar.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved → swinir_top10_bar.png")

In [ ]:
# ==============================
# INSTALL DEPENDENCIES
# ==============================
import subprocess, sys, os, random, glob, math, time

def run(cmd):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"STDERR: {result.stderr}")
    else:
        print(result.stdout.strip())

run("pip install -q timm einops kagglehub scikit-image")

# ==============================
# CLONE SWINIR  (skip if already present)
# ==============================
if not os.path.exists("./SwinIR"):
    print("Cloning SwinIR...")
    run("git clone https://github.com/JingyunLiang/SwinIR.git")
else:
    print("SwinIR already cloned, skipping.")

# verify the models folder exists
swinir_model_path = "./SwinIR/models/network_swinir.py"
if not os.path.exists(swinir_model_path):
    raise FileNotFoundError(
        f"SwinIR model file not found at {swinir_model_path}. "
        "Try deleting the SwinIR folder and re-running."
    )

# add to Python path
SWINIR_ROOT = os.path.abspath("./SwinIR")
if SWINIR_ROOT not in sys.path:
    sys.path.insert(0, SWINIR_ROOT)

print(f"SwinIR path added: {SWINIR_ROOT}")

# quick import check
try:
    from models.network_swinir import SwinIR
    print("✅ SwinIR imported successfully.")
except ModuleNotFoundError as e:
    print(f"❌ Import failed: {e}")
    print("sys.path:", sys.path)

import cv2
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import CosineAnnealingLR
from skimage.metrics import structural_similarity as ssim
# ==============================
# DOWNLOAD DATASETS
# ==============================
import kagglehub

print("Downloading DIV2K...")
div2k_path = kagglehub.dataset_download("soumikrakshit/div2k-high-resolution-images")

print("Downloading Flickr2K...")
flickr2k_path = kagglehub.dataset_download("daehoyang/flickr2k")

# ==============================
# COLLECT ALL IMAGES
# ==============================
def collect_images(root):
    imgs = glob.glob(os.path.join(root, "**/*.png"), recursive=True)
    if not imgs:
        imgs = glob.glob(os.path.join(root, "**/*.jpg"), recursive=True)
    return sorted(imgs)

div2k_imgs    = collect_images(div2k_path)
flickr2k_imgs = collect_images(flickr2k_path)

print(f"DIV2K images   : {len(div2k_imgs)}")
print(f"Flickr2K images: {len(flickr2k_imgs)}")

# ==============================
# SPLIT DATA
# ==============================
random.seed(42)
random.shuffle(div2k_imgs)
test_imgs  = div2k_imgs[:10]
train_imgs = div2k_imgs[10:] + flickr2k_imgs

print(f"Train: {len(train_imgs)} | Test: {len(test_imgs)}")

# ==============================
# HYPERPARAMETERS
# ==============================
PATCH_SIZE  = 64
SCALE       = 4
BATCH_SIZE  = 4      # lower batch = less RAM, safer on Kaggle/Colab
PATCHES_PER = 4      # patches per image per epoch (halved vs before to cut epoch time)
EPOCHS      = 3
LR          = 2e-4
WINDOW_SIZE = 8

# ── CRITICAL for Jupyter / Kaggle / Colab ─────────────────────────────────────
# multiprocessing DataLoader workers deadlock inside ipykernel.
# Always keep num_workers=0 in notebooks.
NUM_WORKERS = 0
# ──────────────────────────────────────────────────────────────────────────────

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# ==============================
# DATA AUGMENTATION
# ==============================
def augment(hr):
    if random.random() > 0.5:
        hr = cv2.flip(hr, 1)
    if random.random() > 0.5:
        hr = cv2.flip(hr, 0)
    k = random.randint(0, 3)
    if k > 0:
        hr = np.rot90(hr, k).copy()   # .copy() avoids negative-stride tensor error
    return hr

# ==============================
# DATASET
# ==============================
class SRDataset(Dataset):
    def __init__(self, img_paths, patch_size=PATCH_SIZE, scale=SCALE,
                 patches_per=PATCHES_PER):
        self.paths      = img_paths
        self.patch_size = patch_size
        self.scale      = scale
        self.ppp        = patches_per

    def __len__(self):
        return len(self.paths) * self.ppp

    def __getitem__(self, idx):
        lr_size = self.patch_size // self.scale
        zero_lr = torch.zeros(3, lr_size,         lr_size)
        zero_hr = torch.zeros(3, self.patch_size, self.patch_size)

        img_path = self.paths[idx % len(self.paths)]
        hr = cv2.imread(img_path)
        if hr is None:
            return zero_lr, zero_hr

        hr = cv2.cvtColor(hr, cv2.COLOR_BGR2RGB)
        h, w = hr.shape[:2]

        if h < self.patch_size or w < self.patch_size:
            return zero_lr, zero_hr

        x        = random.randint(0, w - self.patch_size)
        y        = random.randint(0, h - self.patch_size)
        hr_patch = hr[y:y+self.patch_size, x:x+self.patch_size]
        hr_patch = augment(hr_patch)

        lr_patch = cv2.resize(hr_patch, (lr_size, lr_size),
                              interpolation=cv2.INTER_CUBIC)

        hr_t = torch.from_numpy(hr_patch.astype(np.float32) / 255.).permute(2, 0, 1)
        lr_t = torch.from_numpy(lr_patch.astype(np.float32) / 255.).permute(2, 0, 1)
        return lr_t, hr_t


train_dataset = SRDataset(train_imgs)
train_loader  = DataLoader(
    train_dataset,
    batch_size  = BATCH_SIZE,
    shuffle     = True,
    num_workers = NUM_WORKERS,
    pin_memory  = (device.type == 'cuda'),
    drop_last   = True,
)
print(f"Steps per epoch: {len(train_loader)}")

# ==============================
# MODEL
# Choose 'standard' (paper-spec, 11.8 M params) for best quality,
# or 'lightweight' (faster, ~900 K params) for quick experiments.
# ==============================
MODEL_SIZE = 'standard'

if MODEL_SIZE == 'standard':
    model = SwinIR(
        upscale=SCALE, in_chans=3, img_size=PATCH_SIZE,
        window_size=WINDOW_SIZE, img_range=1.,
        depths=[6, 6, 6, 6, 6, 6], embed_dim=180,
        num_heads=[6, 6, 6, 6, 6, 6], mlp_ratio=2,
        upsampler='pixelshuffle', resi_connection='3conv'
    ).to(device)
else:
    model = SwinIR(
        upscale=SCALE, in_chans=3, img_size=PATCH_SIZE,
        window_size=WINDOW_SIZE, img_range=1.,
        depths=[4, 4, 4, 4], embed_dim=60,
        num_heads=[6, 6, 6, 6], mlp_ratio=2,
        upsampler='pixelshuffle', resi_connection='1conv'
    ).to(device)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model ({MODEL_SIZE}): {total_params/1e6:.1f}M parameters")

# ==============================
# LOSS / OPTIMISER / SCHEDULER
# ==============================
criterion = nn.L1Loss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR, betas=(0.9, 0.99))
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-7)

# ── Modern AMP API — no FutureWarning ─────────────────────────────────────────
use_amp = (device.type == 'cuda')
scaler  = torch.amp.GradScaler('cuda', enabled=use_amp)
# ──────────────────────────────────────────────────────────────────────────────

# ==============================
# TRAINING LOOP
# ==============================
print("\n🚀 Training started...\n")
print(f"{'Epoch':>6} {'Avg Loss':>10} {'Δ Loss':>10} {'LR':>10} {'Time':>8}  {'':>4}")
print("─" * 58)

best_loss    = float('inf')
loss_history = []
LOG_EVERY    = max(1, len(train_loader) // 5)   # heartbeat ~5× per epoch

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0
    steps      = 0
    t0         = time.time()

    for batch_idx, (lr_batch, hr_batch) in enumerate(train_loader):
        lr_batch = lr_batch.to(device, non_blocking=True)
        hr_batch = hr_batch.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast('cuda', enabled=use_amp):
            sr_batch = model(lr_batch)
            loss     = criterion(sr_batch, hr_batch)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
        steps      += 1

        # intra-epoch heartbeat so the notebook shows it's alive
        if (batch_idx + 1) % LOG_EVERY == 0:
            print(f"  ↳ step {batch_idx+1:4d}/{len(train_loader)} "
                  f"| batch loss {loss.item():.5f} "
                  f"| running avg {total_loss/steps:.5f} "
                  f"| {time.time()-t0:.0f}s", flush=True)

    scheduler.step()
    avg_loss  = total_loss / max(1, steps)
    epoch_sec = time.time() - t0
    loss_history.append(avg_loss)

    # delta vs previous epoch (show improvement / worsening)
    if len(loss_history) > 1:
        delta     = avg_loss - loss_history[-2]
        delta_str = f"{delta:+.5f}"
        trend     = "▼" if delta < 0 else ("▲" if delta > 0 else "─")
    else:
        delta_str = "    —    "
        trend     = " "

    # checkpoint flag
    ckpt_flag = ""
    if avg_loss < best_loss:
        best_loss = avg_loss
        torch.save(model.state_dict(), "swinir_best.pth")
        ckpt_flag = "💾"

    current_lr = scheduler.get_last_lr()[0]
    print(f"{epoch:>6d} {avg_loss:>10.5f} {delta_str:>10} "
          f"{current_lr:>10.2e} {epoch_sec:>7.1f}s  {trend} {ckpt_flag}",
          flush=True)

# ── end-of-training summary table ────────────────────────────────────────────
print("\n" + "─" * 58)
print(f"✅ Training complete.")
print(f"   Best loss  : {best_loss:.5f}  (epoch {loss_history.index(min(loss_history))+1})")
print(f"   Final loss : {loss_history[-1]:.5f}")
print(f"   Total drop : {loss_history[0] - loss_history[-1]:+.5f}"
      f"  ({(1 - loss_history[-1]/loss_history[0])*100:.1f}% reduction)")
print("─" * 58)

# ── inline loss curve right after training finishes ──────────────────────────
fig0, ax0 = plt.subplots(figsize=(10, 4))
ax0.plot(range(1, EPOCHS + 1), loss_history,
         color='steelblue', linewidth=2, marker='o', markersize=3, label='Avg L1 loss')
ax0.axhline(min(loss_history), color='#1D9E75', linewidth=1,
            linestyle='--', label=f"Best: {min(loss_history):.5f}")
ax0.fill_between(range(1, EPOCHS + 1), loss_history,
                 min(loss_history), alpha=0.08, color='steelblue')
ax0.set_xlabel("Epoch"); ax0.set_ylabel("L1 Loss")
ax0.set_title("Training Loss per Epoch", fontsize=12)
ax0.legend(); ax0.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("swinir_loss_curve.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved → swinir_loss_curve.png")

# ==============================
# INFERENCE  (with window-size padding)
# ==============================
def inference(model, lr_img):
    h, w  = lr_img.shape[:2]
    pad_h = (WINDOW_SIZE - h % WINDOW_SIZE) % WINDOW_SIZE
    pad_w = (WINDOW_SIZE - w % WINDOW_SIZE) % WINDOW_SIZE

    lr_pad = np.pad(lr_img, ((0, pad_h), (0, pad_w), (0, 0)), mode='reflect')
    lr_t   = (torch.from_numpy(lr_pad.astype(np.float32) / 255.)
              .permute(2, 0, 1).unsqueeze(0).to(device))

    with torch.no_grad():
        with torch.amp.autocast('cuda', enabled=use_amp):
            sr_t = model(lr_t)

    sr = sr_t.squeeze().float().cpu().numpy().transpose(1, 2, 0)
    sr = (sr * 255).clip(0, 255).astype(np.uint8)
    return sr[:h * SCALE, :w * SCALE]

# ==============================
# METRICS
# ==============================
def calc_psnr(img1, img2):
    h = min(img1.shape[0], img2.shape[0])
    w = min(img1.shape[1], img2.shape[1])
    a = img1[:h, :w].astype(np.float32)
    b = img2[:h, :w].astype(np.float32)
    mse = np.mean((a - b) ** 2)
    return float('inf') if mse == 0 else 20 * np.log10(255. / np.sqrt(mse))

def calc_ssim(img1, img2):
    h = min(img1.shape[0], img2.shape[0])
    w = min(img1.shape[1], img2.shape[1])
    return ssim(img1[:h, :w], img2[:h, :w], channel_axis=2, data_range=255)

# ==============================
# EVALUATION
# ==============================
model.eval()
results = []
print("\n🧪 Evaluating on test set...\n")

for img_path in test_imgs:
    hr = cv2.imread(img_path)
    if hr is None:
        continue

    hr    = cv2.cvtColor(hr, cv2.COLOR_BGR2RGB)
    align = SCALE * WINDOW_SIZE
    h, w  = hr.shape[:2]
    hr    = hr[:(h//align)*align, :(w//align)*align]
    h, w  = hr.shape[:2]

    lr      = cv2.resize(hr, (w//SCALE, h//SCALE), interpolation=cv2.INTER_CUBIC)
    bicubic = cv2.resize(lr, (w, h),               interpolation=cv2.INTER_CUBIC)
    sr      = inference(model, lr)

    p_sr  = calc_psnr(hr, sr)
    s_sr  = calc_ssim(hr, sr)
    p_bic = calc_psnr(hr, bicubic)

    results.append({
        "hr": hr, "lr": lr, "sr": sr, "bicubic": bicubic,
        "psnr_sr": p_sr, "ssim_sr": s_sr,
        "psnr_bic": p_bic, "gain": p_sr - p_bic,
        "path": img_path,
    })
    print(f"  {os.path.basename(img_path):<30s} "
          f"SwinIR: {p_sr:.2f} dB | Bicubic: {p_bic:.2f} dB | "
          f"+{p_sr-p_bic:.2f} dB | SSIM: {s_sr:.4f}")

print(f"\n📊 Avg SwinIR PSNR : {np.mean([r['psnr_sr']  for r in results]):.2f} dB")
print(f"📊 Avg Bicubic PSNR: {np.mean([r['psnr_bic'] for r in results]):.2f} dB")
print(f"📊 Avg PSNR Gain   : +{np.mean([r['gain']     for r in results]):.2f} dB")
print(f"📊 Avg SSIM        : {np.mean([r['ssim_sr']  for r in results]):.4f}")

# ==============================
# TOP-10 SELECTION
# ==============================
top_k = min(10, len(results))
top10 = sorted(results, key=lambda x: x["psnr_sr"], reverse=True)[:top_k]

print(f"\n🏆 Top-{top_k} by SwinIR PSNR:")
for i, r in enumerate(top10, 1):
    print(f"  #{i:2d}  {os.path.basename(r['path']):<28s} "
          f"PSNR {r['psnr_sr']:.2f} | Bicubic {r['psnr_bic']:.2f} | "
          f"+{r['gain']:.2f} dB | SSIM {r['ssim_sr']:.4f}")

# ==============================
# HELPER
# ==============================
def centre_crop(img, size=320):
    h, w = img.shape[:2]
    s    = min(h, w, size)
    return img[(h-s)//2:(h-s)//2+s, (w-s)//2:(w-s)//2+s]

# ==============================
# FIGURE 1 — All test results (no loss curve — see swinir_loss_curve.png)
# ==============================
n    = len(results)
fig1, axes1 = plt.subplots(n, 4, figsize=(16, 5*n))
if n == 1:
    axes1 = axes1[np.newaxis, :]   # keep 2-D indexing when n=1
fig1.suptitle("SwinIR — All Test Results", fontsize=13)

for i, r in enumerate(results):
    for ax, img, title in zip(
        axes1[i],
        [r["hr"], r["lr"], r["bicubic"], r["sr"]],
        ["HR", "LR",
         f"Bicubic\n{r['psnr_bic']:.2f} dB",
         f"SwinIR\n{r['psnr_sr']:.2f} dB  SSIM:{r['ssim_sr']:.3f}"]
    ):
        ax.imshow(img); ax.set_title(title, fontsize=9); ax.axis('off')

plt.tight_layout()
plt.savefig("swinir_all_results.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved → swinir_all_results.png")

# ==============================
# FIGURE 2 — Top-10 visual grid
# ==============================
fig2, axes2 = plt.subplots(top_k, 4, figsize=(18, 4.5*top_k))
fig2.suptitle(f"Top-{top_k} Images by SwinIR PSNR  (×{SCALE} SR)",
              fontsize=14, fontweight='bold')

for ax, ct in zip(axes2[0],
                  ["HR (Ground Truth)", "LR (Input)",
                   "Bicubic Baseline",  "SwinIR Output"]):
    ax.set_title(ct, fontsize=11, fontweight='bold', pad=8)

for rank, r in enumerate(top10):
    for col, img in enumerate([r["hr"], r["lr"], r["bicubic"], r["sr"]]):
        ax = axes2[rank][col]
        ax.imshow(centre_crop(img))
        ax.axis('off')

        if col == 0:
            ax.text(0.02, 0.97,
                    f"#{rank+1}  {os.path.basename(r['path'])}",
                    transform=ax.transAxes, fontsize=7.5,
                    color='white', va='top',
                    bbox=dict(boxstyle='round,pad=0.3',
                              facecolor='black', alpha=0.55))
        if col == 2:
            ax.text(0.02, 0.97, f"PSNR: {r['psnr_bic']:.2f} dB",
                    transform=ax.transAxes, fontsize=7.5,
                    color='white', va='top',
                    bbox=dict(boxstyle='round,pad=0.3',
                              facecolor='#555', alpha=0.65))
        if col == 3:
            ax.text(0.02, 0.97,
                    f"PSNR: {r['psnr_sr']:.2f} dB\n"
                    f"vs Bicubic: {r['psnr_bic']:.2f} dB\n"
                    f"Gain: +{r['gain']:.2f} dB\n"
                    f"SSIM: {r['ssim_sr']:.4f}",
                    transform=ax.transAxes, fontsize=7.5,
                    color='white', va='top',
                    bbox=dict(boxstyle='round,pad=0.3',
                              facecolor='#1a6e3c', alpha=0.75))

plt.tight_layout()
plt.savefig("swinir_top10_psnr.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved → swinir_top10_psnr.png")

# ==============================
# FIGURE 3 — PSNR bar chart
# ==============================
fig3, ax3 = plt.subplots(figsize=(12, 5))
names = [f"#{i+1} {os.path.basename(r['path'])[:18]}" for i, r in enumerate(top10)]
p_sw  = [r["psnr_sr"]  for r in top10]
p_bic = [r["psnr_bic"] for r in top10]
x     = np.arange(top_k)
bw    = 0.35

b1 = ax3.bar(x - bw/2, p_bic, bw, label='Bicubic', color='#888780', alpha=0.85)
b2 = ax3.bar(x + bw/2, p_sw,  bw, label='SwinIR',  color='#3266ad', alpha=0.85)

for bar, ps, pb in zip(b2, p_sw, p_bic):
    ax3.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 0.05,
             f"+{ps-pb:.2f}", ha='center', va='bottom',
             fontsize=8, color='#1a5fa8', fontweight='bold')

ax3.set_xticks(x)
ax3.set_xticklabels(names, rotation=30, ha='right', fontsize=9)
ax3.set_ylabel("PSNR (dB)")
ax3.set_title(f"Top-{top_k} — SwinIR vs Bicubic PSNR", fontsize=12)
ax3.legend(); ax3.grid(axis='y', alpha=0.3)
ax3.set_ylim(min(p_bic) - 1, max(p_sw) + 1.5)

plt.tight_layout()
plt.savefig("swinir_top10_bar.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved → swinir_top10_bar.png")

In [ ]:
# ==============================
# INSTALL DEPENDENCIES
# ==============================
import subprocess, sys, os, random, glob, time

def run(cmd):
    subprocess.run(cmd, shell=True)

run("pip install -q timm einops kagglehub scikit-image")

# ==============================
# CLONE SWINIR
# ==============================
if not os.path.exists("./SwinIR"):
    run("git clone https://github.com/JingyunLiang/SwinIR.git")

sys.path.insert(0, os.path.abspath("./SwinIR"))
from models.network_swinir import SwinIR

import cv2
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import CosineAnnealingLR
from skimage.metrics import structural_similarity as ssim
import kagglehub

# ==============================
# DOWNLOAD DATA
# ==============================
div2k_path = kagglehub.dataset_download("soumikrakshit/div2k-high-resolution-images")

def collect_images(root):
    imgs = glob.glob(os.path.join(root, "**/*.png"), recursive=True)
    if not imgs:
        imgs = glob.glob(os.path.join(root, "**/*.jpg"), recursive=True)
    return sorted(imgs)

all_imgs = collect_images(div2k_path)
random.shuffle(all_imgs)

# 🔥 SMALL DATASET (FAST)
train_imgs = all_imgs[:400]
test_imgs  = all_imgs[400:420]

print(f"Train: {len(train_imgs)}, Test: {len(test_imgs)}")

# ==============================
# HYPERPARAMETERS (FAST)
# ==============================
PATCH_SIZE  = 48
SCALE       = 4
BATCH_SIZE  = 8
PATCHES_PER = 2
EPOCHS      = 15
LR          = 2e-4
WINDOW_SIZE = 8

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ==============================
# DATASET (PRELOAD IMAGES 🚀)
# ==============================
class SRDataset(Dataset):
    def __init__(self, img_paths):
        self.images = []
        for p in img_paths:
            img = cv2.imread(p)
            if img is not None:
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                self.images.append(img)

    def __len__(self):
        return len(self.images) * PATCHES_PER

    def __getitem__(self, idx):
        hr = self.images[idx % len(self.images)]

        h, w = hr.shape[:2]
        if h < PATCH_SIZE or w < PATCH_SIZE:
            return torch.zeros(3, PATCH_SIZE//SCALE, PATCH_SIZE//SCALE), \
                   torch.zeros(3, PATCH_SIZE, PATCH_SIZE)

        x = random.randint(0, w - PATCH_SIZE)
        y = random.randint(0, h - PATCH_SIZE)

        hr_patch = hr[y:y+PATCH_SIZE, x:x+PATCH_SIZE]

        lr_patch = cv2.resize(hr_patch,
                              (PATCH_SIZE//SCALE, PATCH_SIZE//SCALE),
                              interpolation=cv2.INTER_CUBIC)

        hr_t = torch.from_numpy(hr_patch.astype(np.float32)/255.).permute(2,0,1)
        lr_t = torch.from_numpy(lr_patch.astype(np.float32)/255.).permute(2,0,1)

        return lr_t, hr_t

train_loader = DataLoader(
    SRDataset(train_imgs),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0
)

# ==============================
# MODEL (LIGHTWEIGHT 🚀)
# ==============================
model = SwinIR(
    upscale=SCALE, in_chans=3, img_size=PATCH_SIZE,
    window_size=WINDOW_SIZE, img_range=1.,
    depths=[4,4,4,4], embed_dim=60,
    num_heads=[6,6,6,6], mlp_ratio=2,
    upsampler='pixelshuffle', resi_connection='1conv'
).to(device)

print("Params:", sum(p.numel() for p in model.parameters())/1e6, "M")

# ==============================
# TRAIN SETUP
# ==============================
criterion = nn.L1Loss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS)

scaler = torch.amp.GradScaler(enabled=(device.type=="cuda"))

# ==============================
# TRAIN LOOP (FAST)
# ==============================
loss_history = []

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    t0 = time.time()

    for lr_img, hr_img in train_loader:
        lr_img = lr_img.to(device)
        hr_img = hr_img.to(device)

        optimizer.zero_grad()

        with torch.amp.autocast(device_type="cuda", enabled=(device.type=="cuda")):
            sr = model(lr_img)
            loss = criterion(sr, hr_img)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()

    scheduler.step()

    avg_loss = total_loss / len(train_loader)
    loss_history.append(avg_loss)

    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {avg_loss:.5f} | Time: {time.time()-t0:.1f}s")

# ==============================
# SAVE MODEL
# ==============================
torch.save(model.state_dict(), "swinir_fast.pth")

# ==============================
# INFERENCE
# ==============================
def inference(model, lr_img):
    lr_t = torch.from_numpy(lr_img.astype(np.float32)/255.).permute(2,0,1).unsqueeze(0).to(device)
    with torch.no_grad():
        sr = model(lr_t)
    sr = sr.squeeze().cpu().numpy().transpose(1,2,0)
    return (sr*255).clip(0,255).astype(np.uint8)

# ==============================
# METRICS
# ==============================
def calc_psnr(a,b):
    mse = np.mean((a-b)**2)
    return 20*np.log10(255./np.sqrt(mse))

# ==============================
# TEST
# ==============================
model.eval()
psnr_list = []

for path in test_imgs:
    hr = cv2.imread(path)
    hr = cv2.cvtColor(hr, cv2.COLOR_BGR2RGB)

    h,w = hr.shape[:2]
    hr = hr[:(h//SCALE)*SCALE, :(w//SCALE)*SCALE]

    lr = cv2.resize(hr, (w//SCALE, h//SCALE))
    sr = inference(model, lr)

    psnr = calc_psnr(hr, sr)
    psnr_list.append(psnr)

print("\nAvg PSNR:", np.mean(psnr_list))

# ==============================
# LOSS PLOT
# ==============================
plt.plot(loss_history)
plt.title("Training Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.show()

In [ ]:
# ==============================
# INSTALL DEPENDENCIES
# ==============================
import subprocess, sys, os, random, glob, time

def run(cmd):
    subprocess.run(cmd, shell=True)

run("pip install -q timm einops kagglehub scikit-image")

# ==============================
# CLONE SWINIR
# ==============================
if not os.path.exists("./SwinIR"):
    run("git clone https://github.com/JingyunLiang/SwinIR.git")

sys.path.insert(0, os.path.abspath("./SwinIR"))
from models.network_swinir import SwinIR

import cv2
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import CosineAnnealingLR
from skimage.metrics import structural_similarity as ssim
import kagglehub

# ==============================
# DOWNLOAD DATA
# ==============================
div2k_path = kagglehub.dataset_download("soumikrakshit/div2k-high-resolution-images")

def collect_images(root):
    imgs = glob.glob(os.path.join(root, "**/*.png"), recursive=True)
    if not imgs:
        imgs = glob.glob(os.path.join(root, "**/*.jpg"), recursive=True)
    return sorted(imgs)

all_imgs = collect_images(div2k_path)
random.shuffle(all_imgs)

train_imgs = all_imgs[:400]
test_imgs  = all_imgs[400:420]

print(f"Train: {len(train_imgs)}, Test: {len(test_imgs)}")

# ==============================
# HYPERPARAMETERS
# ==============================
PATCH_SIZE  = 48
SCALE       = 4
BATCH_SIZE  = 8
PATCHES_PER = 2
EPOCHS      = 30
LR          = 2e-4
WINDOW_SIZE = 8

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ==============================
# DATASET (PRELOAD)
# ==============================
class SRDataset(Dataset):
    def __init__(self, img_paths):
        self.images = []
        for p in img_paths:
            img = cv2.imread(p)
            if img is not None:
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                self.images.append(img)

    def __len__(self):
        return len(self.images) * PATCHES_PER

    def __getitem__(self, idx):
        hr = self.images[idx % len(self.images)]
        h, w = hr.shape[:2]

        if h < PATCH_SIZE or w < PATCH_SIZE:
            return torch.zeros(3, PATCH_SIZE//SCALE, PATCH_SIZE//SCALE), \
                   torch.zeros(3, PATCH_SIZE, PATCH_SIZE)

        x = random.randint(0, w - PATCH_SIZE)
        y = random.randint(0, h - PATCH_SIZE)

        hr_patch = hr[y:y+PATCH_SIZE, x:x+PATCH_SIZE]
        lr_patch = cv2.resize(hr_patch, (PATCH_SIZE//SCALE, PATCH_SIZE//SCALE),
                              interpolation=cv2.INTER_CUBIC)

        hr_t = torch.from_numpy(hr_patch.astype(np.float32)/255.).permute(2,0,1)
        lr_t = torch.from_numpy(lr_patch.astype(np.float32)/255.).permute(2,0,1)

        return lr_t, hr_t

train_loader = DataLoader(SRDataset(train_imgs), batch_size=BATCH_SIZE, shuffle=True)

# ==============================
# MODEL (LIGHTWEIGHT)
# ==============================
model = SwinIR(
    upscale=SCALE, in_chans=3, img_size=PATCH_SIZE,
    window_size=WINDOW_SIZE, img_range=1.,
    depths=[4,4,4,4], embed_dim=60,
    num_heads=[6,6,6,6], mlp_ratio=2,
    upsampler='pixelshuffle', resi_connection='1conv'
).to(device)

# ==============================
# TRAIN
# ==============================
criterion = nn.L1Loss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler = torch.amp.GradScaler(enabled=(device.type=="cuda"))

loss_history = []

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    t0 = time.time()

    for lr_img, hr_img in train_loader:
        lr_img, hr_img = lr_img.to(device), hr_img.to(device)

        optimizer.zero_grad()

        with torch.amp.autocast(device_type="cuda", enabled=(device.type=="cuda")):
            sr = model(lr_img)
            loss = criterion(sr, hr_img)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()

    scheduler.step()
    avg_loss = total_loss / len(train_loader)
    loss_history.append(avg_loss)

    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {avg_loss:.5f} | Time: {time.time()-t0:.1f}s")

torch.save(model.state_dict(), "swinir_fast.pth")

# ==============================
# INFERENCE
# ==============================
def inference(model, lr_img):
    lr_t = torch.from_numpy(lr_img.astype(np.float32)/255.).permute(2,0,1).unsqueeze(0).to(device)
    with torch.no_grad():
        sr = model(lr_t)
    sr = sr.squeeze().cpu().numpy().transpose(1,2,0)
    return (sr*255).clip(0,255).astype(np.uint8)

# ==============================
# METRICS
# ==============================
def calc_psnr(a,b):
    mse = np.mean((a-b)**2)
    return 20*np.log10(255./np.sqrt(mse))

def calc_ssim(a,b):
    return ssim(a, b, channel_axis=2, data_range=255)

# ==============================
# TEST + RESULTS
# ==============================
model.eval()
results, psnr_list, ssim_list = [], [], []

for path in test_imgs:
    hr = cv2.imread(path)
    if hr is None: continue
    hr = cv2.cvtColor(hr, cv2.COLOR_BGR2RGB)

    h,w = hr.shape[:2]
    hr = hr[:(h//SCALE)*SCALE, :(w//SCALE)*SCALE]

    lr = cv2.resize(hr, (w//SCALE, h//SCALE))
    bicubic = cv2.resize(lr, (w,h))
    sr = inference(model, lr)

    psnr_val = calc_psnr(hr, sr)
    ssim_val = calc_ssim(hr, sr)

    psnr_list.append(psnr_val)
    ssim_list.append(ssim_val)

    results.append({"hr":hr,"lr":lr,"bic":bicubic,"sr":sr,
                    "psnr":psnr_val,"ssim":ssim_val,"path":path})

print("\n📊 Avg PSNR:", np.mean(psnr_list))
print("📊 Avg SSIM:", np.mean(ssim_list))

# ==============================
# TOP-10
# ==============================
top10 = sorted(results, key=lambda x:x["psnr"], reverse=True)[:10]

# ==============================
# VISUAL GRID
# ==============================
def crop(img, size=256):
    h,w = img.shape[:2]
    s=min(h,w,size)
    return img[(h-s)//2:(h-s)//2+s,(w-s)//2:(w-s)//2+s]

fig, axes = plt.subplots(len(top10),4,figsize=(16,4*len(top10)))

for i,r in enumerate(top10):
    imgs=[r["hr"],r["lr"],r["bic"],r["sr"]]
    titles=["HR","LR","Bicubic",f"SwinIR\nPSNR:{r['psnr']:.2f}\nSSIM:{r['ssim']:.3f}"]

    for j in range(4):
        axes[i,j].imshow(crop(imgs[j]))
        axes[i,j].set_title(titles[j])
        axes[i,j].axis('off')

plt.tight_layout()
plt.show()

# ==============================
# ZOOMED PATCHES 🔍
# ==============================
fig, axes = plt.subplots(len(top10),4,figsize=(16,4*len(top10)))

for i,r in enumerate(top10):
    h,w = r["hr"].shape[:2]
    cx,cy = w//2, h//2
    s=80

    patches=[
        r["hr"][cy-s:cy+s, cx-s:cx+s],
        r["lr"][cy//SCALE-s//SCALE:cy//SCALE+s//SCALE,
                cx//SCALE-s//SCALE:cx//SCALE+s//SCALE],
        r["bic"][cy-s:cy+s, cx-s:cx+s],
        r["sr"][cy-s:cy+s, cx-s:cx+s]
    ]

    for j in range(4):
        axes[i,j].imshow(patches[j])
        axes[i,j].axis('off')

plt.suptitle("Zoomed Patch Comparison", fontsize=14)
plt.tight_layout()
plt.show()

# ==============================
# BAR CHART
# ==============================
names=[f"{i+1}" for i in range(len(top10))]
p_sr=[r["psnr"] for r in top10]
p_bic=[calc_psnr(r["hr"],r["bic"]) for r in top10]

x=np.arange(len(top10))
plt.bar(x-0.2,p_bic,0.4,label="Bicubic")
plt.bar(x+0.2,p_sr,0.4,label="SwinIR")
plt.legend()
plt.title("PSNR Comparison")
plt.show()

# ==============================
# LOSS CURVE
# ==============================
plt.plot(loss_history)
plt.title("Training Loss")
plt.show()

In [ ]:
# ==============================
# INSTALL + SETUP
# ==============================
import subprocess, sys, os, random, glob, time

def run(cmd):
    subprocess.run(cmd, shell=True)

run("pip install -q timm einops kagglehub scikit-image")

if not os.path.exists("./SwinIR"):
    run("git clone https://github.com/JingyunLiang/SwinIR.git")

sys.path.insert(0, os.path.abspath("./SwinIR"))
from models.network_swinir import SwinIR

import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import CosineAnnealingLR
from skimage.metrics import structural_similarity as ssim
import kagglehub

# ==============================
# DATA
# ==============================
div2k_path = kagglehub.dataset_download("soumikrakshit/div2k-high-resolution-images")

def collect_images(root):
    imgs = glob.glob(os.path.join(root, "**/*.png"), recursive=True)
    if not imgs:
        imgs = glob.glob(os.path.join(root, "**/*.jpg"), recursive=True)
    return sorted(imgs)

all_imgs = collect_images(div2k_path)
random.shuffle(all_imgs)

print("Total images:", len(all_imgs))

if len(all_imgs) < 20:
    raise ValueError("Dataset too small!")

# ==============================
# 🔥 10% TEST SPLIT (FIXED)
# ==============================
test_size = int(0.1 * len(all_imgs))

test_imgs = all_imgs[:test_size]
train_imgs = all_imgs[test_size:]

print(f"Train: {len(train_imgs)}, Test: {len(test_imgs)}")

# ==============================
# HYPERPARAMETERS
# ==============================
PATCH_SIZE  = 48
SCALE       = 4
BATCH_SIZE  = 8
PATCHES_PER = 2
EPOCHS      = 25
LR          = 2e-4
WINDOW_SIZE = 8

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ==============================
# DATASET
# ==============================
class SRDataset(Dataset):
    def __init__(self, img_paths):
        self.images = []
        for p in img_paths:
            img = cv2.imread(p)
            if img is not None:
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                self.images.append(img)

        print("Loaded training images:", len(self.images))

    def __len__(self):
        return len(self.images) * PATCHES_PER

    def __getitem__(self, idx):
        hr = self.images[idx % len(self.images)]
        h, w = hr.shape[:2]

        if h < PATCH_SIZE or w < PATCH_SIZE:
            return torch.zeros(3, PATCH_SIZE//SCALE, PATCH_SIZE//SCALE), \
                   torch.zeros(3, PATCH_SIZE, PATCH_SIZE)

        x = random.randint(0, w - PATCH_SIZE)
        y = random.randint(0, h - PATCH_SIZE)

        hr_patch = hr[y:y+PATCH_SIZE, x:x+PATCH_SIZE]

        lr_patch = cv2.resize(
            hr_patch,
            (PATCH_SIZE//SCALE, PATCH_SIZE//SCALE),
            interpolation=cv2.INTER_CUBIC
        )

        hr_t = torch.from_numpy(hr_patch.astype(np.float32)/255.).permute(2,0,1)
        lr_t = torch.from_numpy(lr_patch.astype(np.float32)/255.).permute(2,0,1)

        return lr_t, hr_t

train_loader = DataLoader(SRDataset(train_imgs), batch_size=BATCH_SIZE, shuffle=True)

# ==============================
# MODEL
# ==============================
model = SwinIR(
    upscale=SCALE,
    in_chans=3,
    img_size=PATCH_SIZE,
    window_size=WINDOW_SIZE,
    img_range=1.,
    depths=[4,4,4,4],
    embed_dim=90,
    num_heads=[6,6,6,6],
    mlp_ratio=2,
    upsampler='pixelshuffle',
    resi_connection='1conv'
).to(device)

# ==============================
# LOSS
# ==============================
l1_loss_fn = nn.L1Loss()

def calc_ssim_tensor(a, b):
    a = a.detach().cpu().numpy().transpose(0,2,3,1)
    b = b.detach().cpu().numpy().transpose(0,2,3,1)
    s = 0
    for i in range(a.shape[0]):
        s += ssim(a[i], b[i], channel_axis=2, data_range=1.0)
    return s / a.shape[0]

optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler = torch.amp.GradScaler(enabled=(device.type=="cuda"))

# ==============================
# TRAIN
# ==============================
loss_history = []

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for lr_img, hr_img in train_loader:
        lr_img, hr_img = lr_img.to(device), hr_img.to(device)

        optimizer.zero_grad()

        with torch.amp.autocast(device_type="cuda", enabled=(device.type=="cuda")):
            bic = F.interpolate(lr_img, scale_factor=4, mode='bicubic', align_corners=False)
            sr = model(lr_img) + bic  # residual learning

            l1 = l1_loss_fn(sr, hr_img)
            ssim_loss = 1 - calc_ssim_tensor(sr, hr_img)

            loss = l1 + 0.1 * ssim_loss

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()

    scheduler.step()
    avg_loss = total_loss / len(train_loader)
    loss_history.append(avg_loss)

    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {avg_loss:.5f}")

# ==============================
# INFERENCE
# ==============================
def inference(model, lr_img):
    lr_t = torch.from_numpy(lr_img.astype(np.float32)/255.).permute(2,0,1).unsqueeze(0).to(device)

    with torch.no_grad():
        bic = F.interpolate(lr_t, scale_factor=4, mode='bicubic', align_corners=False)
        sr = model(lr_t) + bic

    sr = sr.squeeze().cpu().numpy().transpose(1,2,0)
    return (sr*255).clip(0,255).astype(np.uint8)

# ==============================
# METRICS
# ==============================
def calc_psnr(a,b):
    mse = np.mean((a-b)**2)
    return 20*np.log10(255./np.sqrt(mse + 1e-8))

def calc_ssim(a,b):
    return ssim(a, b, channel_axis=2, data_range=255)

# ==============================
# TEST (ROBUST)
# ==============================
model.eval()
results, psnr_list, ssim_list = [], [], []

for path in test_imgs:
    try:
        hr = cv2.imread(path)
        if hr is None:
            continue

        hr = cv2.cvtColor(hr, cv2.COLOR_BGR2RGB)
        h, w = hr.shape[:2]

        if h < 64 or w < 64:
            continue

        hr = hr[:(h//SCALE)*SCALE, :(w//SCALE)*SCALE]
        h, w = hr.shape[:2]

        lr = cv2.resize(hr, (w//SCALE, h//SCALE), interpolation=cv2.INTER_CUBIC)
        bicubic = cv2.resize(lr, (w, h), interpolation=cv2.INTER_CUBIC)

        sr = inference(model, lr)

        if sr.shape != hr.shape:
            continue

        psnr_sr = calc_psnr(hr, sr)
        psnr_bic = calc_psnr(hr, bicubic)
        ssim_sr = calc_ssim(hr, sr)

        psnr_list.append(psnr_sr)
        ssim_list.append(ssim_sr)

        results.append({
            "hr":hr,"lr":lr,"bic":bicubic,"sr":sr,
            "psnr":psnr_sr,"bic_psnr":psnr_bic,
            "ssim":ssim_sr
        })

        print(f"{os.path.basename(path)} | SR:{psnr_sr:.2f} | Bic:{psnr_bic:.2f}")

    except:
        continue

# ==============================
# SAFE CHECK
# ==============================
if len(results) == 0:
    print("❌ No valid test results")
    exit()

print("\n📊 Avg PSNR:", np.mean(psnr_list))
print("📊 Avg SSIM:", np.mean(ssim_list))

# ==============================
# TOP 10
# ==============================
top10 = sorted(results, key=lambda x:x["psnr"], reverse=True)[:10]

# ==============================
# VISUALIZATION
# ==============================
def crop(img, size=256):
    h,w = img.shape[:2]
    s=min(h,w,size)
    return img[(h-s)//2:(h-s)//2+s,(w-s)//2:(w-s)//2+s]

fig, axes = plt.subplots(len(top10),4,figsize=(16,4*len(top10)))

for i,r in enumerate(top10):
    imgs=[r["hr"],r["lr"],r["bic"],r["sr"]]
    titles=["HR","LR","Bicubic",f"SwinIR\nPSNR:{r['psnr']:.2f}\nSSIM:{r['ssim']:.3f}"]

    for j in range(4):
        axes[i,j].imshow(crop(imgs[j]))
        axes[i,j].axis('off')
        axes[i,j].set_title(titles[j])

plt.tight_layout()
plt.show()

# ==============================
# ZOOM PATCHES
# ==============================
fig, axes = plt.subplots(len(top10),4,figsize=(16,4*len(top10)))

for i,r in enumerate(top10):
    h,w = r["hr"].shape[:2]
    cx,cy = w//2, h//2
    s=80

    patches=[
        r["hr"][cy-s:cy+s, cx-s:cx+s],
        r["lr"][cy//SCALE-s//SCALE:cy//SCALE+s//SCALE,
                cx//SCALE-s//SCALE:cx//SCALE+s//SCALE],
        r["bic"][cy-s:cy+s, cx-s:cx+s],
        r["sr"][cy-s:cy+s, cx-s:cx+s]
    ]

    for j in range(4):
        axes[i,j].imshow(patches[j])
        axes[i,j].axis('off')

plt.tight_layout()
plt.show()

# ==============================
# LOSS CURVE
# ==============================
plt.plot(loss_history)
plt.title("Training Loss")
plt.show()

In [ ]:
# ==============================
# INSTALL
# ==============================
import subprocess, os, sys

def run(cmd):
    subprocess.run(cmd, shell=True)

run("pip install -q timm einops kagglehub scikit-image lpips")

# ==============================
# SETUP
# ==============================
if not os.path.exists("./SwinIR"):
    run("git clone https://github.com/JingyunLiang/SwinIR.git")

sys.path.insert(0, os.path.abspath("./SwinIR"))
from models.network_swinir import SwinIR

import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import CosineAnnealingLR
from skimage.metrics import structural_similarity as ssim
import kagglehub
import lpips
import random

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ==============================
# DATA
# ==============================
div2k_path = kagglehub.dataset_download("soumikrakshit/div2k-high-resolution-images")

def collect_images(root):
    imgs = []
    for ext in ["*.png","*.jpg","*.jpeg"]:
        imgs.extend(glob.glob(os.path.join(root,"**",ext),recursive=True))
    return sorted(imgs)

import glob
all_imgs = collect_images(div2k_path)
random.shuffle(all_imgs)

test_size = int(0.1 * len(all_imgs))
test_imgs = all_imgs[:test_size]
train_imgs = all_imgs[test_size:]

print(f"Train: {len(train_imgs)}, Test: {len(test_imgs)}")

# ==============================
# PARAMS
# ==============================
PATCH_SIZE=48
SCALE=4
BATCH_SIZE=8
EPOCHS=20

# ==============================
# DATASET
# ==============================
class SRDataset(Dataset):
    def __init__(self, paths):
        self.imgs=[]
        for p in paths:
            img=cv2.imread(p)
            if img is not None:
                self.imgs.append(cv2.cvtColor(img,cv2.COLOR_BGR2RGB))

    def __len__(self):
        return len(self.imgs)

    def __getitem__(self, idx):
        hr=self.imgs[idx]
        h,w=hr.shape[:2]

        if h<PATCH_SIZE or w<PATCH_SIZE:
            hr=cv2.resize(hr,(PATCH_SIZE,PATCH_SIZE))

        x=random.randint(0,hr.shape[1]-PATCH_SIZE)
        y=random.randint(0,hr.shape[0]-PATCH_SIZE)

        hr=hr[y:y+PATCH_SIZE,x:x+PATCH_SIZE]
        lr=cv2.resize(hr,(PATCH_SIZE//SCALE,PATCH_SIZE//SCALE),interpolation=cv2.INTER_CUBIC)

        hr=torch.from_numpy(hr.astype(np.float32)/255.).permute(2,0,1)
        lr=torch.from_numpy(lr.astype(np.float32)/255.).permute(2,0,1)

        return lr,hr

train_loader=DataLoader(SRDataset(train_imgs),batch_size=BATCH_SIZE,shuffle=True)

# ==============================
# MODEL
# ==============================
model=SwinIR(
    upscale=4,
    in_chans=3,
    img_size=PATCH_SIZE,
    window_size=8,
    img_range=1.,
    depths=[4,4,4,4],
    embed_dim=90,
    num_heads=[6,6,6,6],
    mlp_ratio=2,
    upsampler='pixelshuffle',
    resi_connection='1conv'
).to(device)

optimizer=torch.optim.Adam(model.parameters(),lr=2e-4)
scheduler=CosineAnnealingLR(optimizer,T_max=EPOCHS)
loss_fn=nn.L1Loss()

# ==============================
# LPIPS
# ==============================
lpips_model=lpips.LPIPS(net='alex').to(device)

# ==============================
# TRAIN
# ==============================
loss_hist=[]

for epoch in range(EPOCHS):
    model.train()
    total=0

    for lr,hr in train_loader:
        lr,hr=lr.to(device),hr.to(device)

        optimizer.zero_grad()

        bic=F.interpolate(lr,scale_factor=4,mode='bicubic',align_corners=False)
        sr=model(lr)+bic

        loss=loss_fn(sr,hr)
        loss.backward()
        optimizer.step()

        total+=loss.item()

    scheduler.step()
    loss_hist.append(total/len(train_loader))
    print(f"Epoch {epoch+1}: {loss_hist[-1]:.5f}")

# ==============================
# METRICS (FIXED)
# ==============================
def calc_psnr(a,b):
    return 20*np.log10(255./np.sqrt(np.mean((a-b)**2)+1e-8))

def calc_ssim(a,b):
    return ssim(a,b,channel_axis=2,data_range=255)

def calc_lpips(a,b):
    a=torch.from_numpy(a.astype(np.float32)/127.5-1).permute(2,0,1).unsqueeze(0).to(device)
    b=torch.from_numpy(b.astype(np.float32)/127.5-1).permute(2,0,1).unsqueeze(0).to(device)
    return lpips_model(a,b).item()

# ==============================
# TEST
# ==============================
model.eval()

psnr_s,psnr_b=[],[]
ssim_s,lpips_s=[],[]
results=[]

for path in test_imgs:
    try:
        hr=cv2.imread(path)
        if hr is None: continue

        hr=cv2.cvtColor(hr,cv2.COLOR_BGR2RGB)
        h,w=hr.shape[:2]
        hr=hr[:h//4*4,:w//4*4]

        lr=cv2.resize(hr,(w//4,h//4),interpolation=cv2.INTER_CUBIC)
        bic=cv2.resize(lr,(w,h),interpolation=cv2.INTER_CUBIC)

        lr_t=torch.from_numpy(lr.astype(np.float32)/255.).permute(2,0,1).unsqueeze(0).to(device)

        with torch.no_grad():
            bic_t=F.interpolate(lr_t,scale_factor=4,mode='bicubic',align_corners=False)
            sr=model(lr_t)+bic_t

        sr=sr.squeeze().cpu().numpy().transpose(1,2,0)
        sr=(sr*255).clip(0,255).astype(np.uint8)

        p_s=calc_psnr(hr,sr)
        p_b=calc_psnr(hr,bic)
        s_s=calc_ssim(hr,sr)
        l_s=calc_lpips(hr,sr)

        psnr_s.append(p_s)
        psnr_b.append(p_b)
        ssim_s.append(s_s)
        lpips_s.append(l_s)

        results.append({"hr":hr,"sr":sr,"bic":bic})

    except Exception as e:
        print("Error:",e)

if len(results)==0:
    print("❌ No valid test results")
    exit()

print("\n📊 Avg PSNR:",np.mean(psnr_s))
print("📊 Avg SSIM:",np.mean(ssim_s))
print("📊 Avg LPIPS:",np.mean(lpips_s))

# ==============================
# GRAPHS
# ==============================
plt.plot(psnr_s,label="SwinIR")
plt.plot(psnr_b,label="Bicubic")
plt.legend(); plt.title("PSNR"); plt.show()

plt.plot(ssim_s,label="SSIM")
plt.legend(); plt.title("SSIM"); plt.show()

plt.plot(lpips_s,label="LPIPS")
plt.legend(); plt.title("LPIPS"); plt.show()

gain=np.array(psnr_s)-np.array(psnr_b)
plt.hist(gain,bins=20)
plt.title("PSNR Gain"); plt.show()

plt.scatter(psnr_s,ssim_s)
plt.xlabel("PSNR"); plt.ylabel("SSIM")
plt.title("PSNR vs SSIM"); plt.show()

# ==============================
# ERROR MAP
# ==============================
err=np.abs(results[0]["hr"].astype(np.float32)-results[0]["sr"].astype(np.float32))
plt.imshow(err.astype(np.uint8))
plt.title("Error Map")
plt.show()

# ==============================
# LOSS CURVE
# ==============================
plt.plot(loss_hist)
plt.title("Training Loss")
plt.show()

In [ ]:
# ==============================
# INSTALL + SETUP
# ==============================
import subprocess, sys, os, random, glob

def run(cmd):
    subprocess.run(cmd, shell=True)

run("pip install -q timm einops kagglehub scikit-image pandas")

if not os.path.exists("./SwinIR"):
    run("git clone https://github.com/JingyunLiang/SwinIR.git")

sys.path.insert(0, os.path.abspath("./SwinIR"))
from models.network_swinir import SwinIR

import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import CosineAnnealingLR
from skimage.metrics import structural_similarity as ssim
import kagglehub
import pandas as pd

# ==============================
# DATA
# ==============================
div2k_path = kagglehub.dataset_download("soumikrakshit/div2k-high-resolution-images")

def collect_images(root):
    imgs = glob.glob(os.path.join(root, "**/*.png"), recursive=True)
    if not imgs:
        imgs = glob.glob(os.path.join(root, "**/*.jpg"), recursive=True)
    return sorted(imgs)

all_imgs = collect_images(div2k_path)
random.shuffle(all_imgs)

print("Total images:", len(all_imgs))

test_size = int(0.1 * len(all_imgs))
test_imgs = all_imgs[:test_size]
train_imgs = all_imgs[test_size:]

print(f"Train: {len(train_imgs)}, Test: {len(test_imgs)}")

# ==============================
# HYPERPARAMETERS
# ==============================
PATCH_SIZE  = 48
SCALE       = 4
BATCH_SIZE  = 8
PATCHES_PER = 2
EPOCHS      = 50
LR          = 2e-4
WINDOW_SIZE = 8

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ==============================
# DATASET
# ==============================
class SRDataset(Dataset):
    def __init__(self, img_paths):
        self.images = []
        for p in img_paths:
            img = cv2.imread(p)
            if img is not None:
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                self.images.append(img)

    def __len__(self):
        return len(self.images) * PATCHES_PER

    def __getitem__(self, idx):
        hr = self.images[idx % len(self.images)]
        h, w = hr.shape[:2]

        if h < PATCH_SIZE or w < PATCH_SIZE:
            return torch.zeros(3, PATCH_SIZE//SCALE, PATCH_SIZE//SCALE), \
                   torch.zeros(3, PATCH_SIZE, PATCH_SIZE)

        x = random.randint(0, w - PATCH_SIZE)
        y = random.randint(0, h - PATCH_SIZE)

        hr_patch = hr[y:y+PATCH_SIZE, x:x+PATCH_SIZE]

        lr_patch = cv2.resize(
            hr_patch,
            (PATCH_SIZE//SCALE, PATCH_SIZE//SCALE),
            interpolation=cv2.INTER_CUBIC
        )

        hr_t = torch.from_numpy(hr_patch.astype(np.float32)/255.).permute(2,0,1)
        lr_t = torch.from_numpy(lr_patch.astype(np.float32)/255.).permute(2,0,1)

        return lr_t, hr_t

train_loader = DataLoader(SRDataset(train_imgs), batch_size=BATCH_SIZE, shuffle=True)

# ==============================
# MODEL
# ==============================
model = SwinIR(
    upscale=SCALE,
    in_chans=3,
    img_size=PATCH_SIZE,
    window_size=WINDOW_SIZE,
    img_range=1.,
    depths=[4,4,4,4],
    embed_dim=90,
    num_heads=[6,6,6,6],
    mlp_ratio=2,
    upsampler='pixelshuffle',
    resi_connection='1conv'
).to(device)

# ==============================
# LOSS
# ==============================
l1_loss_fn = nn.L1Loss()

def calc_ssim_tensor(a, b):
    a = a.detach().cpu().numpy().transpose(0,2,3,1)
    b = b.detach().cpu().numpy().transpose(0,2,3,1)
    s = 0
    for i in range(a.shape[0]):
        s += ssim(a[i], b[i], channel_axis=2, data_range=1.0)
    return s / a.shape[0]

optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler = torch.amp.GradScaler(enabled=(device.type=="cuda"))

# ==============================
# TRAIN
# ==============================
loss_history = []

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for lr_img, hr_img in train_loader:
        lr_img, hr_img = lr_img.to(device), hr_img.to(device)

        optimizer.zero_grad()

        with torch.amp.autocast(device_type="cuda", enabled=(device.type=="cuda")):
            bic = F.interpolate(lr_img, scale_factor=4, mode='bicubic', align_corners=False)
            sr = model(lr_img) + bic

            l1 = l1_loss_fn(sr, hr_img)
            ssim_loss = 1 - calc_ssim_tensor(sr.detach(), hr_img.detach())

            loss = l1 + 0.1 * ssim_loss

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()

    scheduler.step()
    avg_loss = total_loss / len(train_loader)
    loss_history.append(avg_loss)

    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {avg_loss:.5f}")

# ==============================
# INFERENCE
# ==============================
def inference(model, lr_img):
    lr_t = torch.from_numpy(lr_img.astype(np.float32)/255.).permute(2,0,1).unsqueeze(0).to(device)

    with torch.no_grad():
        bic = F.interpolate(lr_t, scale_factor=4, mode='bicubic', align_corners=False)
        sr = model(lr_t) + bic

    sr = sr.squeeze().cpu().numpy().transpose(1,2,0)
    return (sr*255).clip(0,255).astype(np.uint8)

# ==============================
# METRICS
# ==============================
def calc_psnr(a,b):
    mse = np.mean((a-b)**2)
    return 20*np.log10(255./np.sqrt(mse + 1e-8))

def calc_ssim(a,b):
    return ssim(a, b, channel_axis=2, data_range=255)

# ==============================
# TEST
# ==============================
model.eval()
results = []

for path in test_imgs:
    try:
        hr = cv2.imread(path)
        if hr is None:
            continue

        hr = cv2.cvtColor(hr, cv2.COLOR_BGR2RGB)
        h, w = hr.shape[:2]

        if h < 64 or w < 64:
            continue

        hr = hr[:(h//SCALE)*SCALE, :(w//SCALE)*SCALE]
        h, w = hr.shape[:2]

        lr = cv2.resize(hr, (w//SCALE, h//SCALE), interpolation=cv2.INTER_CUBIC)
        bic = cv2.resize(lr, (w, h), interpolation=cv2.INTER_CUBIC)
        sr = inference(model, lr)

        if sr.shape != hr.shape:
            continue

        results.append({
            "name": os.path.basename(path),
            "hr": hr,
            "lr": lr,
            "bic": bic,
            "sr": sr,
            "psnr_sr": calc_psnr(hr, sr),
            "ssim_sr": calc_ssim(hr, sr),
            "psnr_bic": calc_psnr(hr, bic),
            "ssim_bic": calc_ssim(hr, bic)
        })

    except:
        continue

# ==============================
# PARAMS
# ==============================
total_params = sum(p.numel() for p in model.parameters())
print(f"\n🧠 Total Parameters: {total_params:,}")

# ==============================
# BEST / WORST (SR + BICUBIC)
# ==============================
best_psnr_sr = max(results, key=lambda x: x["psnr_sr"])
worst_psnr_sr = min(results, key=lambda x: x["psnr_sr"])

best_ssim_sr = max(results, key=lambda x: x["ssim_sr"])
worst_ssim_sr = min(results, key=lambda x: x["ssim_sr"])

best_psnr_bic = max(results, key=lambda x: x["psnr_bic"])
worst_psnr_bic = min(results, key=lambda x: x["psnr_bic"])

best_ssim_bic = max(results, key=lambda x: x["ssim_bic"])
worst_ssim_bic = min(results, key=lambda x: x["ssim_bic"])

# ==============================
# TABLE
# ==============================
table = pd.DataFrame([
    ["Best PSNR (SR)", best_psnr_sr["psnr_sr"], best_psnr_sr["ssim_sr"]],
    ["Worst PSNR (SR)", worst_psnr_sr["psnr_sr"], worst_psnr_sr["ssim_sr"]],
    ["Best SSIM (SR)", best_ssim_sr["psnr_sr"], best_ssim_sr["ssim_sr"]],
    ["Worst SSIM (SR)", worst_ssim_sr["psnr_sr"], worst_ssim_sr["ssim_sr"]],
    ["Best PSNR (Bicubic)", best_psnr_bic["psnr_bic"], best_psnr_bic["ssim_bic"]],
    ["Worst PSNR (Bicubic)", worst_psnr_bic["psnr_bic"], worst_psnr_bic["ssim_bic"]],
    ["Best SSIM (Bicubic)", best_ssim_bic["psnr_bic"], best_ssim_bic["ssim_bic"]],
    ["Worst SSIM (Bicubic)", worst_ssim_bic["psnr_bic"], worst_ssim_bic["ssim_bic"]],
], columns=["Case", "PSNR", "SSIM"])

print("\n📊 FINAL COMPARISON TABLE:")
print(table)

# ==============================
# LOSS CURVE
# ==============================
plt.plot(loss_history)
plt.title("Training Loss")
plt.show()

In [ ]:
# ==============================
# 📊 SWINIR ONLY STATS
# ==============================
psnr_values = [r["psnr_sr"] for r in results]
ssim_values = [r["ssim_sr"] for r in results]

avg_psnr = np.mean(psnr_values)
avg_ssim = np.mean(ssim_values)

# ==============================
# BEST / WORST (SR ONLY)
# ==============================
best_psnr_sr = max(results, key=lambda x: x["psnr_sr"])
worst_psnr_sr = min(results, key=lambda x: x["psnr_sr"])

best_ssim_sr = max(results, key=lambda x: x["ssim_sr"])
worst_ssim_sr = min(results, key=lambda x: x["ssim_sr"])

# ==============================
# 📋 FINAL TABLE (SWINIR ONLY)
# ==============================
table = pd.DataFrame([
    ["Best PSNR", best_psnr_sr["psnr_sr"], best_psnr_sr["ssim_sr"]],
    ["Worst PSNR", worst_psnr_sr["psnr_sr"], worst_psnr_sr["ssim_sr"]],
    ["Best SSIM", best_ssim_sr["psnr_sr"], best_ssim_sr["ssim_sr"]],
    ["Worst SSIM", worst_ssim_sr["psnr_sr"], worst_ssim_sr["ssim_sr"]],
    ["Average", avg_psnr, avg_ssim],
], columns=["Case", "PSNR", "SSIM"])

print("\n📊 SWINIR PERFORMANCE TABLE:")
print(table)

# Optional pretty display
try:
    from IPython.display import display
    display(table)
except:
    pass

In [ ]:
# ==============================
# INSTALL + SETUP
# ==============================
import subprocess, sys, os, random, glob, time

def run(cmd):
    subprocess.run(cmd, shell=True)

run("pip install -q timm einops kagglehub scikit-image")

if not os.path.exists("./SwinIR"):
    run("git clone https://github.com/JingyunLiang/SwinIR.git")

sys.path.insert(0, os.path.abspath("./SwinIR"))
from models.network_swinir import SwinIR

import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import CosineAnnealingLR
from skimage.metrics import structural_similarity as ssim
import kagglehub

# ==============================
# DATA
# ==============================
div2k_path = kagglehub.dataset_download("soumikrakshit/div2k-high-resolution-images")

def collect_images(root):
    imgs = glob.glob(os.path.join(root, "**/*.png"), recursive=True)
    if not imgs:
        imgs = glob.glob(os.path.join(root, "**/*.jpg"), recursive=True)
    return sorted(imgs)

all_imgs = collect_images(div2k_path)
random.shuffle(all_imgs)

print("Total images:", len(all_imgs))

# 10% test split
test_size = int(0.1 * len(all_imgs))
test_imgs = all_imgs[:test_size]
train_imgs = all_imgs[test_size:]

print(f"Train: {len(train_imgs)}, Test: {len(test_imgs)}")

# ==============================
# HYPERPARAMETERS
# ==============================
PATCH_SIZE = 48
SCALE = 4
BATCH_SIZE = 8
PATCHES_PER = 2
EPOCHS = 25
LR = 2e-4
WINDOW_SIZE = 8

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ==============================
# DATASET
# ==============================
class SRDataset(Dataset):
    def __init__(self, paths):
        self.images = []
        for p in paths:
            img = cv2.imread(p)
            if img is not None:
                self.images.append(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))

    def __len__(self):
        return len(self.images) * PATCHES_PER

    def __getitem__(self, idx):
        hr = self.images[idx % len(self.images)]
        h, w = hr.shape[:2]

        if h < PATCH_SIZE or w < PATCH_SIZE:
            hr = cv2.resize(hr, (PATCH_SIZE, PATCH_SIZE))

        x = random.randint(0, w - PATCH_SIZE)
        y = random.randint(0, h - PATCH_SIZE)

        hr_patch = hr[y:y+PATCH_SIZE, x:x+PATCH_SIZE]
        lr_patch = cv2.resize(hr_patch, (PATCH_SIZE//SCALE, PATCH_SIZE//SCALE), interpolation=cv2.INTER_CUBIC)

        hr_t = torch.from_numpy(hr_patch.astype(np.float32)/255.).permute(2,0,1)
        lr_t = torch.from_numpy(lr_patch.astype(np.float32)/255.).permute(2,0,1)

        return lr_t, hr_t

train_loader = DataLoader(SRDataset(train_imgs), batch_size=BATCH_SIZE, shuffle=True)

# ==============================
# MODEL
# ==============================
model = SwinIR(
    upscale=SCALE,
    in_chans=3,
    img_size=PATCH_SIZE,
    window_size=WINDOW_SIZE,
    img_range=1.,
    depths=[4,4,4,4],
    embed_dim=90,
    num_heads=[6,6,6,6],
    mlp_ratio=2,
    upsampler='pixelshuffle',
    resi_connection='1conv'
).to(device)

# ==============================
# TRAINING SETUP
# ==============================
loss_fn = nn.L1Loss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS)

loss_history = []

# ==============================
# TRAIN
# ==============================
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for lr_img, hr_img in train_loader:
        lr_img, hr_img = lr_img.to(device), hr_img.to(device)

        optimizer.zero_grad()

        bic = F.interpolate(lr_img, scale_factor=4, mode='bicubic', align_corners=False)
        sr = model(lr_img) + bic

        loss = loss_fn(sr, hr_img)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    scheduler.step()
    avg_loss = total_loss / len(train_loader)
    loss_history.append(avg_loss)

    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {avg_loss:.5f}")

# ==============================
# INFERENCE
# ==============================
def inference(model, lr_img):
    lr_t = torch.from_numpy(lr_img.astype(np.float32)/255.).permute(2,0,1).unsqueeze(0).to(device)

    with torch.no_grad():
        bic = F.interpolate(lr_t, scale_factor=4, mode='bicubic', align_corners=False)
        sr = model(lr_t) + bic

    sr = sr.squeeze().cpu().numpy().transpose(1,2,0)
    return (sr*255).clip(0,255).astype(np.uint8)

# ==============================
# METRICS
# ==============================
def calc_psnr(a,b):
    mse = np.mean((a-b)**2)
    return 20*np.log10(255./np.sqrt(mse + 1e-8))

def calc_ssim(a,b):
    return ssim(a, b, channel_axis=2, data_range=255)

# ==============================
# TEST
# ==============================
model.eval()
results, psnr_list, ssim_list = [], [], []

for path in test_imgs:
    try:
        hr = cv2.imread(path)
        if hr is None:
            continue

        hr = cv2.cvtColor(hr, cv2.COLOR_BGR2RGB)
        h, w = hr.shape[:2]

        if h < 64 or w < 64:
            continue

        hr = hr[:(h//SCALE)*SCALE, :(w//SCALE)*SCALE]
        h, w = hr.shape[:2]

        lr = cv2.resize(hr, (w//SCALE, h//SCALE), interpolation=cv2.INTER_CUBIC)
        bic = cv2.resize(lr, (w, h), interpolation=cv2.INTER_CUBIC)

        sr = inference(model, lr)

        if sr.shape != hr.shape:
            continue

        psnr_sr = calc_psnr(hr, sr)
        ssim_sr = calc_ssim(hr, sr)

        psnr_list.append(psnr_sr)
        ssim_list.append(ssim_sr)

        results.append({
            "hr": hr,
            "lr": lr,
            "bic": bic,
            "sr": sr,
            "psnr": psnr_sr,
            "ssim": ssim_sr
        })

    except:
        continue

if len(results) == 0:
    print("❌ No valid test results")
    exit()

print("\n📊 Avg PSNR:", np.mean(psnr_list))
print("📊 Avg SSIM:", np.mean(ssim_list))

# ==============================
# TOP 10
# ==============================
top10 = sorted(results, key=lambda x: x["psnr"], reverse=True)[:10]

# ==============================
# VISUALIZATION (FINAL)
# ==============================
def center_crop(img, size=256):
    h, w = img.shape[:2]
    s = min(h, w, size)
    return img[(h-s)//2:(h-s)//2+s, (w-s)//2:(w-s)//2+s]

def zoom_patch(img, cx, cy, size=80):
    return img[cy-size:cy+size, cx-size:cx+size]

fig, axes = plt.subplots(len(top10), 5, figsize=(18, 4 * len(top10)))

for i, r in enumerate(top10):

    hr = r["hr"]
    lr = r["lr"]
    sr = r["sr"]

    h, w = hr.shape[:2]
    cx, cy = w // 2, h // 2

    lr_up = cv2.resize(lr, (w, h), interpolation=cv2.INTER_NEAREST)

    hr_patch = zoom_patch(hr, cx, cy)
    lr_patch = zoom_patch(lr_up, cx, cy)
    sr_patch = zoom_patch(sr, cx, cy)

    axes[i, 0].imshow(center_crop(hr))
    axes[i, 0].set_title("HR")
    axes[i, 0].axis('off')

    axes[i, 1].imshow(center_crop(lr_up))
    axes[i, 1].set_title("LR")
    axes[i, 1].axis('off')

    axes[i, 2].imshow(lr_patch)
    axes[i, 2].set_title("Zoom LR")
    axes[i, 2].axis('off')

    axes[i, 3].imshow(sr_patch)
    axes[i, 3].set_title("Zoom SR")
    axes[i, 3].axis('off')

    axes[i, 4].imshow(center_crop(sr))
    axes[i, 4].set_title(f"SwinIR\nPSNR:{r['psnr']:.2f}\nSSIM:{r['ssim']:.3f}")
    axes[i, 4].axis('off')

plt.tight_layout()
plt.show()

# ==============================
# LOSS CURVE
# ==============================
plt.plot(loss_history)
plt.title("Training Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.show()